In [ ]:
import copy
import math
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
import plotly.subplots as sbplt
from plotly.subplots import make_subplots

from experiments_2024 import DATASETS_PATH, IMAGE_PATH
from experiments_2024.datasets import (
    load_zones,
    load_weather,
    load_building,
    pull_from_dataset,
)
from experiments_2024.zone_level_analysis import (
    base,
    cleaning,
    viz,
    regression_functions,
    clustering,
)

from experiments_2024 import constants
import statsmodels.api as sm

In [ ]:
pd.set_option("display.max_rows", 100000000)

# Helper vars

In [ ]:
KWH_PER_TON_COOLING = 3.5168528
WH_PER_BTU = 0.293071
CFM_to_CMH = 0.0283168 / 60  # m3/hr
PROJECTS = ["OFF-2", "OFF-3", "OFF-4", "OFF-5", "OFF-6", "OFF-7"]
SUMMER_START = pd.Timestamp("05-01-2024")
SUMMER_END = pd.Timestamp("10-01-2024")
ONLY_BUSINESS_HOURS = True
NO_WEEKENDS = True
TRIALS = ["All-Zone", "Trial 1", "Trial 2", "Trial 3"]
X_LINE_120 = np.linspace(0.000001, 120, 200)
X_LINE_100 = np.linspace(0.000001, 100, 200)

In [ ]:
# for multi-plots
WIDTH = 1000
HEIGHT = 600
NUM_COLS = 2
TITLE_SIZE = 36
TEXT_SIZE = 28
LEGEND_SIZE = 32
ANNOTATION_SIZE = 32
MARKER_SIZE = 14
HORIZONTAL_SPACING = 0.175
VERTICAL_SPACING = 0.125
LEGEND_SPACING = -0.075

In [ ]:
SF = pd.Series(index=PROJECTS)
SF["OFF-2"] = 135556
SF["OFF-3"] = 28420
SF["OFF-4"] = 169619
SF["OFF-5"] = 84501
SF["OFF-6"] = 105465
SF["OFF-7"] = 63439

In [ ]:
ALL_ZONES_LISTS = {}
for project in PROJECTS:
    ALL_ZONES_LISTS[project] = {}
    all_zones = list(
        set(
            cleaning.clean_df(
                load_zones("2024", project, "zone-temps"),
                this_var="zone-temps",
                only_business_hours=False,
                no_weekends=False,
            ).columns
        ).union(
            set(
                cleaning.clean_df(
                    load_zones("2024", project, "zone-coolsp"),
                    this_var="zone-coolsp",
                    only_business_hours=False,
                    no_weekends=False,
                ).columns
            ).union(
                set(
                    cleaning.clean_df(
                        load_zones("2024", project, "zone-tloads"),
                        this_var="zone-tloads",
                        only_business_hours=False,
                        no_weekends=False,
                    ).columns
                )
            )
        )
    )
    vavs = [zone for zone in all_zones if "VAV" in zone.upper()]
    FCs = [zone for zone in all_zones if "FC" in zone.upper()]
    ALL_ZONES_LISTS[project]["All"] = all_zones
    ALL_ZONES_LISTS[project]["VAVs"] = vavs
    ALL_ZONES_LISTS[project]["FCs"] = FCs

In [ ]:
EXCLUDED_ZONES_LISTS = {}
for project in PROJECTS:
    EXCLUDED_ZONES_LISTS[project] = {}
    ezs = pd.read_csv(
        f"{DATASETS_PATH}/csvs/2024_experiment_csvs/{project}_excluded_zones.csv"
    )
    trials = ["Trial 1", "Trial 2", "Trial 3"]
    sets = {c: set(ezs[c].dropna().astype(str)) for c in trials}
    excluded_all = set.intersection(*sets.values())
    excluded_any = set.union(*sets.values())
    excluded_some = excluded_any - excluded_all

    EXCLUDED_ZONES_LISTS[project]["Any Trial"] = list(
        excluded_all.intersection(set(ALL_ZONES_LISTS[project]["VAVs"]))
    )
    EXCLUDED_ZONES_LISTS[project]["All Trials"] = list(
        excluded_all.intersection(set(ALL_ZONES_LISTS[project]["VAVs"]))
    )
    EXCLUDED_ZONES_LISTS[project]["Some Trials"] = list(
        excluded_some.intersection(set(ALL_ZONES_LISTS[project]["VAVs"]))
    )

In [ ]:
DOMINANT_ZONES_LISTS = {}
for project in PROJECTS:
    DOMINANT_ZONES_LISTS[project] = {}
    dz_df = pd.read_csv(
        f"{DATASETS_PATH}/csvs/2024_experiment_csvs/{project}_dominant_zones.csv"
    ).set_index("VAV")
    common = list(
        set(list(dz_df.index)).intersection(set(ALL_ZONES_LISTS[project]["VAVs"]))
    )
    dz_df = dz_df.loc[common, :]
    DOMINANT_ZONES_LISTS[project]["All-Zone"] = common
    for trial in ["Trial 1", "Trial 2", "Trial 3"]:
        if project == "OFF-7" and trial == "Trial 3":
            continue
        DOMINANT_ZONES_LISTS[project][trial] = list(
            dz_df[trial][dz_df[trial] == 1].index
        )

In [ ]:
ZONES_WRITTEN_TO_LISTS = {}
for project in PROJECTS:
    ZONES_WRITTEN_TO_LISTS[project] = {}
    for trial in ["All-Zone", "Trial 1", "Trial 2", "Trial 3"]:
        if project == "OFF-7" and trial == "Trial 3":
            continue
        ZONES_WRITTEN_TO_LISTS[project][trial] = list(
            set(DOMINANT_ZONES_LISTS[project][trial])
            - set(EXCLUDED_ZONES_LISTS[project]["Any Trial"])
        )

In [ ]:
ZONE_GROUPS_NUM_TO_STR = {
    0: "All-Zone",
    1: "Trial 1",
    2: "Trial 2",
    3: "Trial 3",
    4: "Excluded",
}

ZONE_GROUPS_STR_TO_NUM = {v: k for k, v in ZONE_GROUPS_NUM_TO_STR.items()}

ZONE_GROUPS_NUM_TO_COLOR = {
    0: "Silver",
    1: "ForestGreen",
    2: "DarkOrange",
    3: "Firebrick",
    4: "Dimgray",
}

In [ ]:
ZONE_GROUPS = {}
for project in PROJECTS:
    this_group = pd.Series(0, index=ALL_ZONES_LISTS[project]["VAVs"])

    # trial 1-3
    for trial in ["Trial 1", "Trial 2", "Trial 3"]:
        if project == "OFF-7" and trial == "Trial 3":
            continue
        these_zones = list(
            set(ALL_ZONES_LISTS[project]["VAVs"]).intersection(
                set(DOMINANT_ZONES_LISTS[project][trial])
            )
        )
        this_group[these_zones] = ZONE_GROUPS_STR_TO_NUM[trial]

    # excluded all trials
    these_zones = list(
        set(ALL_ZONES_LISTS[project]["VAVs"]).intersection(
            set(EXCLUDED_ZONES_LISTS[project]["Any Trial"])
        )
    )
    this_group[these_zones] = 4

    ZONE_GROUPS[project] = this_group.to_frame()

In [ ]:
count = pd.DataFrame(index=PROJECTS, columns=list(ZONE_GROUPS_STR_TO_NUM.keys()))
for project in PROJECTS:
    for i in list(range(len(ZONE_GROUPS_NUM_TO_STR))):
        count.loc[project, ZONE_GROUPS_NUM_TO_STR[i]] = len(
            ZONE_GROUPS[project][ZONE_GROUPS[project] == i].dropna()
        )

count_no_off7 = count.iloc[:-1, :]

count["Total"] = count.sum(axis=1)
count.loc["Total", :] = count.sum(axis=0)
count_norm = 100 * count.div(count["Total"], axis=0).apply(
    pd.to_numeric, errors="coerce"
).round(3)

count_no_off7["Total"] = count_no_off7.sum(axis=1)
count_no_off7.loc["Total", :] = count_no_off7.sum(axis=0)
count_no_off7_norm = 100 * count_no_off7.div(count_no_off7["Total"], axis=0).apply(
    pd.to_numeric, errors="coerce"
).round(3)

In [ ]:
# count_norm

In [ ]:
# count_no_off7_norm

In [ ]:
PORTION_ZONES = pd.DataFrame(
    index=PROJECTS, columns=["All-Zone", "Trial 1", "Trial 2", "Trial 3"]
)
for project in PROJECTS:
    for trial in ["All-Zone", "Trial 1", "Trial 2", "Trial 3"]:
        if project == "OFF-7" and trial == "Trial 3":
            continue
        PORTION_ZONES.loc[project, trial] = (
            100
            * len(ZONES_WRITTEN_TO_LISTS[project][trial])
            / len(ALL_ZONES_LISTS[project]["VAVs"])
        )

# Helper functions

In [ ]:
def load_utility_data(
    projects,
    time_range,
    year="2024",
    field="C",
):
    utility = load_building(year, field).loc[time_range[0] : time_range[1], projects]
    if field == "C":
        utility = utility * KWH_PER_TON_COOLING  # to kWh
    if field == "H":
        utility = utility * WH_PER_BTU  # to kWh
    return utility

In [ ]:
def get_building_wide_results(df, T, mode, summary_statistic="Mean", return_covs=False):
    if isinstance(df, dict):
        projects = list(df.keys())
    else:
        projects = list(df.columns)
    # place to hold results
    cols = [
        "Delta All-Zone",
        "Delta Trial 1",
        "Delta Trial 2",
        "Delta Trial 3",
        "Delta High All-Zone",
        "Delta High Trial 1",
        "Delta High Trial 2",
        "Delta High Trial 3",
        "Delta Low All-Zone",
        "Delta Low Trial 1",
        "Delta Low Trial 2",
        "Delta Low Trial 3",
        "Slope OAT",
        "Slope All-Zone",
        "Slope Trial 1",
        "Slope Trial 2",
        "Slope Trial 3",
        "Slope End Summer",
        "Std Err OAT",
        "Std Err All-Zone",
        "Std Err Trial 1",
        "Std Err Trial 2",
        "Std Err Trial 3",
        "Std Err End Summer",
        "P-Value OAT",
        "P-Value All-Zone",
        "P-Value Trial 1",
        "P-Value Trial 2",
        "P-Value Trial 3",
        "P-Value End Summer",
    ]
    summary = pd.DataFrame(np.nan, index=projects, columns=cols)
    covs = {}
    for project in projects:
        # binary df
        binary_df = regression_functions.get_2024_binary_df(
            project,
            freq="daily",
            baseline_column="Control",
            no_weekends=NO_WEEKENDS,
            control_for_weekends=(not NO_WEEKENDS),
            control_for_summer=regression_functions.CONTROL_FOR_SUMMER[project],
            off4_all_zone="adjust",
            off7_trial_3="drop",
        )
        y = df[project].dropna()
        if isinstance(y, pd.Series):
            y = y.to_frame()
        y, this_T = base.trim_to_common_elements(
            [y, copy.deepcopy(T)], clean_cols=False, clean_idx=True
        )
        if return_covs:
            reg_results, this_cov = regression_functions.general_Delta_fn(
                df=y,
                T=this_T["temperature"],
                binary=binary_df,
                mode=mode,
                summary_statistic=summary_statistic,
                return_cov=True,
            )
            covs[project] = this_cov[project]
        else:
            reg_results = regression_functions.general_Delta_fn(
                df=y,
                T=this_T["temperature"],
                binary=binary_df,
                mode=mode,
                summary_statistic=summary_statistic,
                return_cov=False,
            )
        these_cols = [col for col in reg_results.columns if col in cols]
        summary.loc[project, cols] = reg_results.loc[project, these_cols]
    if return_covs:
        return summary, covs
    return summary

In [ ]:
def process_summary_df(summary):
    projects = list(summary.index)
    y = summary.loc[
        :,
        [
            "Delta All-Zone",
            "Delta Trial 1",
            "Delta Trial 2",
            "Delta Trial 3",
        ],
    ]
    down = (
        y.values
        - summary.loc[
            :,
            [
                "Delta Low All-Zone",
                "Delta Low Trial 1",
                "Delta Low Trial 2",
                "Delta Low Trial 3",
            ],
        ]
    )
    up = (
        summary.loc[
            :,
            [
                "Delta High All-Zone",
                "Delta High Trial 1",
                "Delta High Trial 2",
                "Delta High Trial 3",
            ],
        ]
        - y.values
    )
    for df in [y, up, down]:
        df.columns = TRIALS
    return y, up, down

In [ ]:
def process_summary_df_norm(summary, covs=None):
    y = summary.loc[
        :,
        [
            "Delta All-Zone",
            "Delta Trial 1",
            "Delta Trial 2",
            "Delta Trial 3",
        ],
    ]
    y.columns = TRIALS
    y_norm = copy.deepcopy(y)
    for col in TRIALS:
        y_norm[col] = y[col] / y["All-Zone"]
    if covs is None:
        return 100 * y_norm

    y_norm_var = pd.DataFrame(index=PROJECTS, columns=TRIALS)
    for project in PROJECTS:
        b_all_zone = summary.loc[project, "Slope All-Zone"]
        var_all_zone = (summary.loc[project, "Std Err All-Zone"]) ** 2
        for trial in TRIALS:
            if project == "OFF-7" and trial == "Trial 3":
                continue
            b_i = summary.loc[project, f"Slope {trial}"]
            var_bi = (summary.loc[project, f"Std Err {trial}"]) ** 2
            cov = covs[project].loc["All-Zone", trial]
            y_norm_var.loc[project, trial] = (
                (var_bi / b_all_zone**2)
                + ((b_i**2) * (var_all_zone) / b_all_zone**4)
                - (2 * b_i * cov / (b_all_zone**3))
            )
    y_norm_se = y_norm_var ** (1 / 2)
    y_norm_up = 1.96 * y_norm_se
    y_norm_down = 1.96 * y_norm_se
    y_norm_up["All-Zone"] = 0
    y_norm_down["All-Zone"] = 0
    return 100 * y_norm, 100 * y_norm_up, 100 * y_norm_down

In [ ]:
def get_trial_data(
    this_var,
    this_var_clean=None,
    projects=PROJECTS,
    this_test="Mean",
    start_date=SUMMER_START,
    end_date=SUMMER_END,
    only_business_hours=ONLY_BUSINESS_HOURS,
    no_weekends=NO_WEEKENDS,
    SI_units=True,
    clean_underlying=False,
):
    if this_var_clean is None:
        this_var_clean = this_var

    y = pull_from_dataset(
        "2024",
        projects,
        this_var,
        clean_data=clean_underlying,
        resample_data=True,
    )

    projects_trial3 = copy.deepcopy(projects)
    if "OFF-7" in projects_trial3:
        projects_trial3.remove("OFF-7")
    y_trial3 = {key: value for key, value in y.items() if key in projects_trial3}

    # control
    y_control = cleaning.clean_dfs(
        dfs=y,
        this_var=this_var_clean,
        remove_FCUs=True,
        only_business_hours=only_business_hours,
        no_weekends=no_weekends,
        start_date=start_date,
        end_date=end_date,
        hourly_filter=cleaning.get_experiment_hourly_filter(
            projects=projects,
            filter_columns=["Control"],
            no_weekends=no_weekends,
            freq="hourly",
        ),
        SI_units=SI_units,
    )
    if this_test is not None:
        y_control = base.run_passive_test_on_dfs(
            y_control, this_test=this_test, col_name="Control"
        )

    # trial 1
    y_trial1 = cleaning.clean_dfs(
        dfs=y,
        this_var=this_var_clean,
        remove_FCUs=True,
        only_business_hours=only_business_hours,
        no_weekends=no_weekends,
        start_date=start_date,
        end_date=end_date,
        hourly_filter=cleaning.get_experiment_hourly_filter(
            projects=projects,
            filter_columns=["Trial 1"],
            no_weekends=no_weekends,
            freq="hourly",
        ),
        SI_units=SI_units,
    )
    if this_test is not None:
        y_trial1 = base.run_passive_test_on_dfs(
            y_trial1, this_test=this_test, col_name="Trial 1"
        )

    # trial 2
    y_trial2 = cleaning.clean_dfs(
        dfs=y,
        this_var=this_var_clean,
        remove_FCUs=True,
        only_business_hours=only_business_hours,
        no_weekends=no_weekends,
        start_date=start_date,
        end_date=end_date,
        hourly_filter=cleaning.get_experiment_hourly_filter(
            projects=projects,
            filter_columns=["Trial 2"],
            no_weekends=no_weekends,
            freq="hourly",
        ),
        SI_units=SI_units,
    )
    if this_test is not None:
        y_trial2 = base.run_passive_test_on_dfs(
            y_trial2, this_test=this_test, col_name="Trial 2"
        )

    # trial 3
    y_trial3 = cleaning.clean_dfs(
        dfs=y_trial3,
        this_var=this_var_clean,
        remove_FCUs=True,
        only_business_hours=only_business_hours,
        no_weekends=no_weekends,
        start_date=start_date,
        end_date=end_date,
        hourly_filter=cleaning.get_experiment_hourly_filter(
            projects=projects_trial3,
            filter_columns=["Trial 3"],
            no_weekends=no_weekends,
            freq="hourly",
        ),
        SI_units=SI_units,
    )
    if this_test is not None:
        y_trial3 = base.run_passive_test_on_dfs(
            y_trial3, this_test=this_test, col_name="Trial 3"
        )
    y_trial3["OFF-7"] = copy.deepcopy(y_control["OFF-7"])
    y_trial3["OFF-7"].loc[:, :] = np.nan
    if this_test is not None:
        y_trial3["OFF-7"].columns = ["Trial 3"]

    # all zones
    y_allzones = cleaning.clean_dfs(
        dfs=y,
        this_var=this_var_clean,
        remove_FCUs=True,
        only_business_hours=only_business_hours,
        no_weekends=no_weekends,
        start_date=start_date,
        end_date=end_date,
        hourly_filter=cleaning.get_experiment_hourly_filter(
            projects=projects,
            filter_columns=["All-Zone"],
            no_weekends=no_weekends,
            freq="hourly",
        ),
        SI_units=SI_units,
    )
    if this_test is not None:
        y_allzones = base.run_passive_test_on_dfs(
            y_allzones, this_test=this_test, col_name="All-Zone"
        )
    return y_control, y_trial1, y_trial2, y_trial3, y_allzones

In [ ]:
def make_tot_df(this_dict):
    tot_df = []
    for project in this_dict.keys():
        this_df = copy.deepcopy(this_dict[project])
        this_df.index = project + " " + this_df.index.astype(str)
        tot_df.append(this_df)
    tot_df = pd.concat(tot_df, axis=0)
    return tot_df

In [ ]:
def create_lorenz(CRs):
    if isinstance(CRs, pd.DataFrame):
        CRs = CRs.iloc[:, 0]
    CRs = CRs.sort_values(ascending=False)
    CRs = CRs.cumsum()
    CRs = CRs / CRs[-1]
    return CRs

In [ ]:
def gini(x):
    x = np.array(x)
    x_sorted = np.sort(x)
    n = len(x)
    cumx = np.cumsum(x_sorted)
    return (n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n

In [ ]:
def plot_2024_experiment_summary(
    deltas,
    portion_zones,
    deltas_high=None,
    deltas_low=None,
    mode="lines+markers",
    line_legend=None,
    y_axis_title="Y Axis Title",
    secondary_y_axis_title="Y Axis Title",
    x_axis_title="% of Spaces Adjusted",
    y_range=None,
    second_y_range=None,
    x_range=[0, 100],
    secondary_variables=None,
    title_size=34,
    text_size=24,
    legend_size=28,
    horizontal_spacing=viz.HORIZONTAL_SPACING,
    vertical_spacing=viz.VERTICAL_SPACING,
    width=viz.GRAPH_WIDTH,
    height=viz.GRAPH_HEIGHT,
    num_cols=viz.GRAPH_NUM_COLS,
    append_zero=True,
    grid_color="LightGray",
    y_zerolinecolor="Black",
    line_width=3,
    marker_size=10,
    whisker_len=10,
    error_thickness=2,
    show_legend=True,
    x_zerolinecolor="White",
):
    if isinstance(deltas, pd.DataFrame):
        deltas = {" ": deltas}
        deltas_high = {" ": deltas_high}
        deltas_low = {" ": deltas_low}

    keys = list(deltas.keys())
    trials = list(deltas[keys[0]].columns)
    projects = list(deltas[keys[0]].index)
    titles = projects

    if len(projects) >= num_cols:
        graph_num_cols = num_cols
    else:
        graph_num_cols = len(projects)
    graph_num_rows = math.ceil((len(projects)) / graph_num_cols)

    specs = [
        [
            {"secondary_y": secondary_variables is not None}
            for _ in range(graph_num_cols)
        ]
        for _ in range(graph_num_rows)
    ]

    fig = sbplt.make_subplots(
        rows=graph_num_rows,
        cols=graph_num_cols,
        subplot_titles=titles,
        horizontal_spacing=horizontal_spacing,
        vertical_spacing=vertical_spacing,
        specs=specs,
    )

    for k in range(len(keys)):
        key = keys[k]
        graph_r = 0
        graph_c = 0
        for i in range(len(projects)):
            if graph_c == graph_num_cols:
                graph_c = 0
                graph_r += 1
            project = projects[i]

            this_y = list(deltas[key].loc[project, trials].dropna().values)
            if deltas_high is not None and deltas_high[key] is not None:
                this_error_up = list(
                    deltas_high[key].loc[project, trials].dropna().values
                )
            else:
                this_error_up = None
            if deltas_low is not None and deltas_low[key] is not None:
                this_error_down = list(
                    deltas_low[key].loc[project, trials].dropna().values
                )
            else:
                this_error_down = None
            x = list(portion_zones.loc[project, trials].dropna().values)

            if append_zero:
                x.append(0)
                if deltas_high is not None:
                    this_error_up.append(0)
                if deltas_low is not None:
                    this_error_down.append(0)

            if line_legend is not None and "name" in line_legend:
                name = line_legend["name"][key]
            else:
                name = key

            if len(keys) == 1:
                color = "Black"
            elif line_legend is not None and "color" in line_legend:
                color = line_legend["color"][key]
            else:
                color = viz.COLORS[k]

            if len(keys) == 1:
                opacity = 1
            elif line_legend is not None and "opacity" in line_legend:
                opacity = line_legend["opacity"][key]
            else:
                opacity = 1

            if len(keys) == 1:
                shape = "circle"
            elif line_legend is not None and "shape" in line_legend:
                shape = line_legend["shape"][key]
            else:
                shape = "circle"

            if len(keys) == 1:
                dash = "solid"
            elif line_legend is not None and "dash" in line_legend:
                dash = line_legend["dash"][key]
            else:
                dash = "solid"

            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=this_y,
                    mode=mode,
                    error_y=dict(
                        type="data",
                        array=this_error_up,
                        arrayminus=this_error_down,
                        width=whisker_len,
                        thickness=error_thickness,
                    ),
                    line=dict(color=color, width=line_width, dash=dash),
                    opacity=opacity,
                    name=name,
                    showlegend=False,
                    marker_size=marker_size,
                    marker_symbol=shape,
                ),
                row=graph_r + 1,
                col=graph_c + 1,
                secondary_y=(
                    key in secondary_variables
                    if secondary_variables is not None
                    else False
                ),
            )
            graph_c += 1

    # legend
    if show_legend:
        for k in range(len(keys)):
            key = keys[k]
            if line_legend is not None and "name" in line_legend:
                name = line_legend["name"][key]
            else:
                name = key
            if name is None:
                continue

            if len(keys) == 1:
                color = "Black"
            elif line_legend is not None and "color" in line_legend:
                color = line_legend["color"][key]
            else:
                color = viz.COLORS[k]

            if len(keys) == 1:
                opacity = 1
            elif line_legend is not None and "opacity" in line_legend:
                opacity = line_legend["opacity"][key]
            else:
                opacity = 1

            if len(keys) == 1:
                shape = "circle"
            elif line_legend is not None and "shape" in line_legend:
                shape = line_legend["shape"][key]
            else:
                shape = "circle"

            if len(keys) == 1:
                dash = "solid"
            elif line_legend is not None and "dash" in line_legend:
                dash = line_legend["dash"][key]
            else:
                dash = "solid"

            fig.add_trace(
                go.Scatter(
                    x=[np.nan],
                    y=[np.nan],
                    mode=mode,
                    line=dict(color=color, width=3, dash=dash),
                    opacity=opacity,
                    name=name,
                    showlegend=True,
                    marker_size=marker_size,
                    marker_symbol=shape,
                ),
            )

    # format
    fig = viz.update_fig_formatting(
        fig,
        y_axis_title=y_axis_title,
        x_axis_title=x_axis_title,
        width=width * graph_num_cols,
        height=height * graph_num_rows,
        title_size=title_size,
        text_size=text_size,
        legend_size=legend_size,
        y_range=y_range,
        x_range=x_range,
        y_zerolinecolor=y_zerolinecolor,
        grid_color=grid_color,
    )

    fig = fig.update_layout(
        legend=dict(
            x=0.5,
            y=-0.125,
            xanchor="center",
            yanchor="top",
            orientation="h",
        ),
    )
    if secondary_variables is not None:
        fig.update_yaxes(
            title_text=secondary_y_axis_title,
            secondary_y=True,
        )
        fig.update_yaxes(range=second_y_range, secondary_y=True)
    return fig

# Testbed

In [ ]:
testbed = pd.DataFrame(
    index=[
        "Year of Construction",
        "Year of Last Retrofit",
        "1000m2",
        "# of AHs",
        "# of VAVs + FCs",
        "# of VAVs",
        "# of FCs",
        "Summer Daily Energy Demand (kWh/m2/day)",
        "% Cooling Demand",
        "% Electric Demand",
        "% Heating Demand",
        # "# of Excluded VAVs",
        "Total Experiment Days",
        "# of Control Days",
        "# of Trial 1 Days",
        "# of Trial 2 Days",
        "# of Trial 3 Days",
        "# of All-Zone Days",
        "Daytime OAT Control Days [°C]",
        "Daytime OAT Trial 1 Days [°C]",
        "Daytime OAT Trial 2 Days [°C]",
        "Daytime OAT Trial 3 Days [°C]",
        "Daytime OAT All-Zone Days [°C]",
        "% of VAVs Adjusted, All-Zone",
        "% of VAVs Adjusted, Trial 1",
        "% of VAVs Adjusted, Trial 2",
        "% of VAVs Adjusted, Trial 3",
        "% of VAVs Excluded From All Trials",
        "% of VAVs Excluded From Some Trials",
    ],
    columns=PROJECTS,
)
testbed["TOTAL"] = np.nan

## Year of construction

In [ ]:
testbed.loc["Year of Construction", "OFF-2"] = 1996
testbed.loc["Year of Construction", "OFF-3"] = 1893
testbed.loc["Year of Construction", "OFF-4"] = 1996
testbed.loc["Year of Construction", "OFF-5"] = 2015
testbed.loc["Year of Construction", "OFF-6"] = 1998
testbed.loc["Year of Construction", "OFF-7"] = 1900

## Year of last retrofit

In [ ]:
testbed.loc["Year of Last Retrofit", "OFF-2"] = 2021
testbed.loc["Year of Last Retrofit", "OFF-3"] = 2015
testbed.loc["Year of Last Retrofit", "OFF-4"] = 2015
testbed.loc["Year of Last Retrofit", "OFF-5"] = np.nan
testbed.loc["Year of Last Retrofit", "OFF-6"] = 2012
testbed.loc["Year of Last Retrofit", "OFF-7"] = 2017

## m2

In [ ]:
for project in PROJECTS:
    testbed.loc["1000m2", project] = SF[project] * base.M2_PER_SF / 1000
testbed.loc["1000m2", "TOTAL"] = SF.sum() * base.M2_PER_SF / 1000

## Number of equipment

In [ ]:
for project in PROJECTS:
    testbed.loc["# of AHs", project] = len(
        list(
            cleaning.clean_df(load_zones("2023", project, "ahu-dat"), "ahu-dat").columns
        )
    )

    # vavs
    testbed.loc["# of VAVs", project] = len(ALL_ZONES_LISTS[project]["VAVs"])
    # FCs
    testbed.loc["# of FCs", project] = len(ALL_ZONES_LISTS[project]["FCs"])

for row in [
    "# of VAVs",
    "# of FCs",
    "# of AHs",
]:
    testbed.loc[row, "TOTAL"] = testbed.loc[row, PROJECTS].sum()

testbed.loc["# of VAVs + FCs", :] = (
    testbed.loc["# of VAVs", :] + testbed.loc["# of FCs", :]
)

## # of zones adjusted and excluded

In [ ]:
# Adjusted
for project in PROJECTS:
    for trial in TRIALS:
        if project == "OFF-7" and trial == "Trial 3":
            continue
        testbed.loc[f"% of VAVs Adjusted, {trial}", project] = len(
            ZONES_WRITTEN_TO_LISTS[project][trial]
        )

for trial in TRIALS:
    testbed.loc[f"% of VAVs Adjusted, {trial}", "TOTAL"] = testbed.loc[
        f"% of VAVs Adjusted, {trial}", :
    ].sum()

In [ ]:
# excluded
for project in PROJECTS:
    testbed.loc["% of VAVs Excluded From All Trials", project] = len(
        EXCLUDED_ZONES_LISTS[project]["All Trials"]
    )
    testbed.loc["% of VAVs Excluded From Some Trials", project] = len(
        EXCLUDED_ZONES_LISTS[project]["Some Trials"]
    )
testbed.loc["% of VAVs Excluded From All Trials", "TOTAL"] = testbed.loc[
    "% of VAVs Excluded From All Trials", PROJECTS
].sum()
testbed.loc["% of VAVs Excluded From Some Trials", "TOTAL"] = testbed.loc[
    "% of VAVs Excluded From Some Trials", PROJECTS
].sum()

In [ ]:
# normalize
these_projects = copy.deepcopy(PROJECTS)
these_projects.append("TOTAL")
for project in these_projects:
    tot = testbed.loc["# of VAVs", project]
    testbed.loc["% of VAVs Excluded From All Trials", project] = (
        testbed.loc["% of VAVs Excluded From All Trials", project] / tot
    )
    testbed.loc["% of VAVs Excluded From Some Trials", project] = (
        testbed.loc["% of VAVs Excluded From Some Trials", project] / tot
    )
    for trial in TRIALS:
        if trial == "Trial 3" and project == "TOTAL":
            tot = tot - testbed.loc["# of VAVs", "OFF-7"]
        testbed.loc[f"% of VAVs Adjusted, {trial}", project] = (
            testbed.loc[f"% of VAVs Adjusted, {trial}", project] / tot
        )

## Experiment days

In [ ]:
OAT = cleaning.clean_df(
    df=load_weather("2024")["temperature"].to_frame(),
    this_var="weather-oat",
    no_weekends=True,
    only_business_hours=True,
    SI_units=True,
)

OAT = OAT.groupby(OAT.index.date).mean().iloc[:, 0]

In [ ]:
these_trials = copy.deepcopy(TRIALS)
these_trials.append("Control")

for project in PROJECTS:
    binary_df = regression_functions.get_2024_binary_df(
        project,
        freq="daily",
        baseline_column="Control",
        drop_baseline_column=False,
        no_weekends=True,
        control_for_weekends=False,
        control_for_summer=False,
        off4_all_zone="adjust",
        off7_trial_3="drop",
    )
    for trial in these_trials:
        if project == "OFF-7" and trial == "Trial 3":
            continue
        testbed.loc[f"# of {trial} Days", project] = len(
            binary_df[binary_df[trial] == 1].index
        )
        testbed.loc[f"Daytime OAT {trial} Days [°C]", project] = OAT[
            binary_df[binary_df[trial] == 1].index
        ].sum()

for trial in these_trials:
    testbed.loc[f"# of {trial} Days", "TOTAL"] = testbed.loc[
        f"# of {trial} Days", :
    ].sum()
    testbed.loc[f"Daytime OAT {trial} Days [°C]", "TOTAL"] = testbed.loc[
        f"Daytime OAT {trial} Days [°C]", :
    ].sum()
    testbed.loc[f"Daytime OAT {trial} Days [°C]", :] = (
        (
            testbed.loc[f"Daytime OAT {trial} Days [°C]", :]
            / testbed.loc[f"# of {trial} Days", :]
        )
        .astype(float)
        .round(1)
    )


for project in testbed.columns:
    testbed.loc["Total Experiment Days", project] = (
        testbed.loc[f"# of Control Days", project]
        + testbed.loc[f"# of Trial 1 Days", project]
        + testbed.loc[f"# of Trial 2 Days", project]
        + testbed.loc[f"# of All-Zone Days", project]
    )
    if project != "OFF-7":
        testbed.loc["Total Experiment Days", project] += testbed.loc[
            f"# of Trial 3 Days", project
        ]

## Cooling

In [ ]:
cooling = cleaning.clean_df(
    df=load_building("2023", "C"),
    this_var="building-cooling",
    start_date=pd.Timestamp("05-01-2023"),
    end_date=pd.Timestamp("10-01-2023"),
    only_business_hours=False,
    no_weekends=True,
)
cooling = cooling[PROJECTS]

In [ ]:
fig = viz.make_time_series(cooling, y_axis_title="Cooling (tons)")

In [ ]:
# fig

In [ ]:
cooling = cooling * base.MW_PER_TON * (10**3)
cooling["TOTAL"] = cooling.sum(axis=1, skipna=True)
cooling = cooling.groupby(cooling.index.date).sum()
for project in list(cooling.columns):
    testbed.loc["Average Cooling Demand (kWh/m2/day)", project] = cooling[
        project
    ].mean() / (1000 * testbed.loc["1000m2", project])

## Heating

In [ ]:
heating = cleaning.clean_df(
    df=load_building("2023", "H"),
    this_var="building-heating",
    start_date=pd.Timestamp("05-01-2023"),
    end_date=pd.Timestamp("10-01-2023"),
    only_business_hours=False,
    no_weekends=True,
)
heating = heating[PROJECTS]

In [ ]:
fig = viz.make_time_series(heating, y_axis_title="Heating (kBtu)")

In [ ]:
# fig

In [ ]:
heating = heating * base.WH_PER_BTU
heating["TOTAL"] = heating.sum(axis=1, skipna=True)
heating = heating.groupby(heating.index.date).sum()
heating[heating == 0] = np.nan  # edge case
for project in list(heating.columns):
    testbed.loc["Average Heating Demand (kWh/m2/day)", project] = heating[
        project
    ].mean() / (1000 * testbed.loc["1000m2", project])

## Electricity

In [ ]:
electricity = cleaning.clean_df(
    df=load_building("2023", "E"),
    this_var="building-electricity",
    start_date=pd.Timestamp("05-01-2023"),
    end_date=pd.Timestamp("10-01-2023"),
    only_business_hours=False,
    no_weekends=True,
)
electricity = electricity[PROJECTS]
electricity[electricity > 1000] = np.nan
electricity["OFF-2"][electricity["OFF-2"] > 400] = np.nan
electricity["OFF-3"][electricity["OFF-3"] > 20] = np.nan
electricity["OFF-4"][electricity["OFF-4"] > 250] = np.nan
electricity["OFF-5"][electricity["OFF-5"] > 250] = np.nan
electricity["OFF-6"][electricity["OFF-6"] > 300] = np.nan
electricity["OFF-7"][electricity["OFF-7"] > 200] = np.nan

In [ ]:
fig = viz.make_time_series(
    electricity, y_axis_title="Electricity (kW)", force_same_yaxes=False
)

In [ ]:
# fig

In [ ]:
electricity["TOTAL"] = electricity.sum(axis=1, skipna=True)
electricity = electricity.groupby(electricity.index.date).sum()
for project in list(electricity.columns):
    testbed.loc["Average Electric Demand (kWh/m2/day)", project] = electricity[
        project
    ].mean() / (1000 * testbed.loc["1000m2", project])

## Total energy

In [ ]:
testbed.loc["Summer Daily Energy Demand (kWh/m2/day)", PROJECTS] = (
    testbed.loc["Average Cooling Demand (kWh/m2/day)", PROJECTS]
    + testbed.loc["Average Heating Demand (kWh/m2/day)", PROJECTS]
    + testbed.loc["Average Electric Demand (kWh/m2/day)", PROJECTS]
)
testbed.loc["Summer Daily Energy Demand (kWh/m2/day)", "TOTAL"] = (
    (
        testbed.loc["Summer Daily Energy Demand (kWh/m2/day)", PROJECTS]
        * testbed.loc["1000m2", PROJECTS]
        * 1000
    ).sum()
) / (testbed.loc["1000m2", PROJECTS].sum() * 1000)

for utility in ["Cooling", "Heating", "Electric"]:
    testbed.loc[f"% {utility} Demand", :] = (
        (
            testbed.loc[f"Average {utility} Demand (kWh/m2/day)", :]
            / testbed.loc["Summer Daily Energy Demand (kWh/m2/day)", :]
        )
        .astype(float)
        .round(3)
    )
    testbed.drop(index=f"Average {utility} Demand (kWh/m2/day)", inplace=True)

In [ ]:
# testbed

# Choosing zones

In [ ]:
CALIBRATION_START = {
    "OFF-2": pd.Timestamp("04-29-2024"),
    "OFF-7": pd.Timestamp("04-29-2024"),
    "OFF-4": pd.Timestamp("04-29-2024"),
    "OFF-5": pd.Timestamp("04-29-2024"),
    "OFF-6": pd.Timestamp("04-29-2024"),
    "OFF-3": pd.Timestamp("06-14-2024"),
}
CALIBRATION_END = {
    "OFF-2": pd.Timestamp("05-31-2024"),
    "OFF-7": pd.Timestamp("05-31-2024"),
    "OFF-4": pd.Timestamp("05-31-2024"),
    "OFF-5": pd.Timestamp("05-31-2024"),
    "OFF-6": pd.Timestamp("06-19-2024"),
    "OFF-3": pd.Timestamp("06-19-2024"),
}
ADJUSTMENTS_START = {
    "OFF-2": pd.Timestamp("06-03-2024"),
    "OFF-7": pd.Timestamp("06-03-2024"),
    "OFF-4": pd.Timestamp("06-03-2024"),
    "OFF-5": pd.Timestamp("06-03-2024"),
    "OFF-6": pd.Timestamp("06-24-2024"),
    "OFF-3": pd.Timestamp("06-19-2024"),
}
ADJUSTMENTS_END = {
    "OFF-2": pd.Timestamp("06-19-2024"),
    "OFF-7": pd.Timestamp("06-19-2024"),
    "OFF-4": pd.Timestamp("06-19-2024"),
    "OFF-5": pd.Timestamp("06-19-2024"),
    "OFF-6": pd.Timestamp("07-10-2024"),
    "OFF-3": pd.Timestamp("06-25-2024"),
}
TRIAL_1_START = {
    "OFF-2": pd.Timestamp("06-19-2024"),
    "OFF-5": pd.Timestamp("06-19-2024"),
    "OFF-4": pd.Timestamp("06-19-2024"),
    "OFF-7": pd.Timestamp("06-19-2024"),
    "OFF-3": pd.Timestamp("06-27-2024"),
    "OFF-6": pd.Timestamp("06-27-2024"),
}
TRIAL_1_END = {
    "OFF-2": pd.Timestamp("07-13-2024"),
    "OFF-5": pd.Timestamp("07-13-2024"),
    "OFF-4": pd.Timestamp("07-13-2024"),
    "OFF-7": pd.Timestamp("07-13-2024"),
    "OFF-3": pd.Timestamp("07-17-2024"),
    "OFF-6": pd.Timestamp("08-02-2024"),
}
TRIAL_2_END = {
    "OFF-2": pd.Timestamp("08-02-2024"),
    "OFF-5": pd.Timestamp("08-02-2024"),
    "OFF-4": pd.Timestamp("08-02-2024"),
    "OFF-7": pd.Timestamp("08-02-2024"),
    "OFF-3": pd.Timestamp("08-02-2024"),
    "OFF-6": pd.Timestamp("08-23-2024"),
}

In [ ]:
trial1_date_table = pd.DataFrame(index=PROJECTS, columns=["Start Date", "End Date"])
for project in PROJECTS:
    trial1_date_table.loc[project, "Start Date"] = CALIBRATION_START[project].strftime(
        "%Y-%m-%d"
    )
    trial1_date_table.loc[project, "End Date"] = CALIBRATION_END[project].strftime(
        "%Y-%m-%d"
    )

adjust_date_table = pd.DataFrame(index=PROJECTS, columns=["Start Date", "End Date"])
for project in PROJECTS:
    adjust_date_table.loc[project, "Start Date"] = ADJUSTMENTS_START[project].strftime(
        "%Y-%m-%d"
    )
    adjust_date_table.loc[project, "End Date"] = ADJUSTMENTS_END[project].strftime(
        "%Y-%m-%d"
    )

trial2_date_table = pd.DataFrame(index=PROJECTS, columns=["Start Date", "End Date"])
for project in PROJECTS:
    trial2_date_table.loc[project, "Start Date"] = TRIAL_1_START[project].strftime(
        "%Y-%m-%d"
    )
    trial2_date_table.loc[project, "End Date"] = TRIAL_1_END[project].strftime(
        "%Y-%m-%d"
    )

these_projects = copy.deepcopy(PROJECTS)
these_projects.remove("OFF-7")
trial3_date_table = pd.DataFrame(
    index=these_projects, columns=["Start Date", "End Date"]
)
for project in these_projects:
    trial3_date_table.loc[project, "Start Date"] = TRIAL_1_START[project].strftime(
        "%Y-%m-%d"
    )
    trial3_date_table.loc[project, "End Date"] = TRIAL_2_END[project].strftime(
        "%Y-%m-%d"
    )

In [ ]:
CRs = pull_from_dataset(
    "2024", PROJECTS, "zone-simple_cooling_requests", resample_data=True
)

tloads = pull_from_dataset("2024", PROJECTS, "zone-tloads", resample_data=True)

deviation = pull_from_dataset(
    "2024", PROJECTS, "zone-deviation_coolsp", resample_data=True
)

In [ ]:
control_filter = cleaning.get_experiment_hourly_filter(
    projects=PROJECTS,
    filter_columns=["Control"],
    freq="hourly",
    no_weekends=True,
    off7_trial_3="drop",
)

trial1_filter = cleaning.get_experiment_hourly_filter(
    PROJECTS,
    filter_columns={
        "OFF-2": ["Trial 1"],
        "OFF-5": ["Trial 1"],
        "OFF-4": ["Trial 1", "Trial 1 (Poor Control)"],
        "OFF-7": ["Trial 1"],
        "OFF-3": ["Trial 1"],
        "OFF-6": ["Trial 1"],
    },
    no_weekends=True,
)

trial2_filter = cleaning.get_experiment_hourly_filter(
    PROJECTS, ["Trial 2"], no_weekends=True
)

allzone_filter = cleaning.get_experiment_hourly_filter(
    PROJECTS,
    filter_columns={
        "OFF-2": ["All-Zone"],
        "OFF-5": ["All-Zone"],
        "OFF-4": ["All-Zone", "All-Zone (Poor Control)"],
        "OFF-7": [
            "All-Zone",
        ],
        "OFF-3": ["All-Zone"],
        "OFF-6": ["All-Zone"],
    },
    no_weekends=True,
)

experiment_filter = cleaning.get_experiment_hourly_filter(
    projects=PROJECTS,
    filter_columns=["Trial 1", "Trial 2", "Trial 3"],
    freq="hourly",
    no_weekends=True,
    off7_trial_3="drop",
)

## Trial 1

In [ ]:
tloads_calibration = cleaning.clean_dfs(
    tloads,
    this_var="zone-tloads",
    start_date=CALIBRATION_START,
    end_date=CALIBRATION_END,
    only_business_hours=True,
    no_weekends=True,
    SI_units=False,
    remove_FCUs=True,
)
deviation_calibration = cleaning.clean_dfs(
    deviation,
    this_var="zone-deviation_coolsp",
    start_date=CALIBRATION_START,
    end_date=CALIBRATION_END,
    only_business_hours=True,
    no_weekends=True,
    SI_units=False,
    remove_FCUs=True,
)

tloads_calibration = base.run_passive_test_on_dfs(
    tloads_calibration,
    this_test="Mean",
    col_name="Tloads",
)
deviation_calibration = base.run_passive_test_on_dfs(
    deviation_calibration,
    this_test="Mean",
    col_name="Deviation",
)
deviation_calibration["OFF-6"].loc["VAV 1-26", :] = np.nan  # edge case

tloads_calibration, deviation_calibration = base.make_common_index(
    [tloads_calibration, deviation_calibration]
)

In [ ]:
trial1_slices_tload = {
    "OFF-2": 9.5,
    "OFF-3": 26,
    "OFF-4": 14,
    "OFF-5": 5,
    "OFF-6": 9,
    "OFF-7": 6.9,
}
trial1_slices_dev = {
    "OFF-2": -1.5,
    "OFF-3": -0.1,
    "OFF-4": -1.4,
    "OFF-5": -1.7,
    "OFF-6": -1.8,
    "OFF-7": -1.61,
}

In [ ]:
trial1_thresh_table = pd.DataFrame(
    index=PROJECTS, columns=["Terminal Load [%]", "T - CSP [°F]"]
)

for project in PROJECTS:
    trial1_thresh_table.loc[project, "Terminal Load [%]"] = trial1_slices_tload[project]
    trial1_thresh_table.loc[project, "T - CSP [°F]"] = trial1_slices_dev[project]

In [ ]:
# trial1_thresh_table

In [ ]:
manual_adjustments = {
    "OFF-2": [
        "VAV 1-323",
        "VAV 1-442",
        "VAV 2-504-B",
        "VAV 2-504-C",
        "VAV 2-319",
        "VAV 3-392",
        "VAV 3-376-A",
        "VAV 3-376-B",
        "VAV 3-380-A",
        "VAV 3-380-B",
        "VAV 3-386",
        "VAV 3-284A",
        "VAV 3-374",
        "VAV 3-372",
        "VAV 3-370",
        "VAV 3-188",
        "VAV 3-289-B",
        "VAV 3-289-A",
        "VAV 3-282",
        "VAV 2-101",
        "VAV 3-161A",
        "VAV 3-161B",
    ],
    "OFF-3": [
        "VAV 212",
        "VAV 206",
        "VAV C14",
    ],
    "OFF-4": [
        "VAV 3-10-04",
        "VAV 4-10-02",
        "VAV 3-10-02",
        "VAV 4-10-04",
        "VAV 3-02-03",
        "VAV 3-02-13",
        "VAV 3-02-10",
        "VAV 3-02-07",
        "VAV 2-03-05",
        "VAV 1-05-13",
        "VAV 1-05-06",
        "VAV 1-05-01",
        "VAV 1-06-09",
        "VAV 4-09-03",
        "VAV 3-09-06",
        "VAV 3-09-02",
        "VAV 4-09-05",
        "VAV 4-09-01",
        "VAV 2-04-07",
        "VAV 3-04-10",
    ],
    "OFF-5": [
        "VAV 12-09",
        "VAV 12-05",
        "VAV 13-06-1",
        "VAV 13-06-2",
        "VAV 12-02",
        "VAV 12-14",
        "VAV 11-07",
        "VAV 11-08",
        "VAV 11-10",
        "VAV 11-11",
        "VAV 33-17",
        "VAV 33-22",
        "VAV 33-26",
        "VAV 33-27",
        "VAV 33-25",
        "VAV 33-29",
        "VAV 33-19",
        "VAV 33-09",
        "VAV 32-04",
        "VAV 33-36",
        "VAV 33-01",
        "VAV 1B-03",
    ],
    "OFF-6": [
        "VAV 3-54",
        "VAV 3-08",
        "VAV 3-17",
        "VAV 1-27",
        "VAV 3-22",
        "VAV 2-60",
        "VAV 1-41",
        "VAV B-45",
        "VAV 2-47",
        "VAV 2-49",
        "VAV B-38",
        "VAV 1-31",
        "VAV 1-26",
        "VAV 3-25",
        "VAV B-46",
        "VAV 3-53",
        "VAV 2-55",
        "VAV 1-29",
        "VAV 3-39",
        "VAV 3-29",
    ],
    "OFF-7": ["VAV0-15", "VAV2-19", "VAV3-10"],
}

In [ ]:
trial1_unadj = clustering.run_2D_clustering_on_dict(
    tloads_calibration,
    deviation_calibration,
    slice1=trial1_slices_tload,
    slice2=trial1_slices_dev,
    mapping={0: 1, 1: 0, 2: 0, 3: 0, 4: 0},
)
trial1 = copy.deepcopy(trial1_unadj)
for project in PROJECTS:
    trial1[project].loc[manual_adjustments[project], :] = 1
    trial1[project].loc[EXCLUDED_ZONES_LISTS[project]["Any Trial"]] = 2

In [ ]:
fig = viz.make_scatter_plot(
    y_data=tloads_calibration,
    x_data=deviation_calibration,
    color_data=trial1,
    color_legend={
        "name": {0: "Unadjusted in Trial 1", 1: "Adjusted in Trial 1", 2: "Excluded"},
        "color": {0: "Silver", 1: "ForestGreen", 2: "Dimgray"},
    },
    color_legend_order=[2, 0, 1],
    color_and_shape=True,
    shape_legend={
        "shape": {0: "circle", 1: "circle", 2: "x"},
    },
    y_axis_title="Terminal Load [%]",
    x_axis_title="T - CSP [°F]",
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE,
    marker_size=MARKER_SIZE,
    horizontal_spacing=HORIZONTAL_SPACING,
    vertical_spacing=VERTICAL_SPACING,
)
fig = fig.update_layout(
    margin=dict(t=100),
    title=dict(
        text="Decisions for Trial 1",
        x=0.5,  # center horizontally
        xanchor="center",
        y=0.98,  # move up/down
        yanchor="top",
        font=dict(size=TITLE_SIZE),
    ),
    legend=dict(
        x=0.5,
        y=LEGEND_SPACING,
        xanchor="center",
        yanchor="top",
        orientation="h",
    ),
)

p = 0
r = 1
c = 1
for project in PROJECTS:
    if c == 3:
        r += 1
        c = 1
    fig.add_hline(
        y=trial1_slices_tload[project],
        row=r,
        col=c,
        line_dash="dash",
        line_color="black",
        line_width=4,
    )
    fig.add_vline(
        x=trial1_slices_dev[project],
        row=r,
        col=c,
        line_dash="dash",
        line_color="black",
        line_width=4,
    )

    c += 1
    p += 1

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/FigureB1.png")

## Trial 2

In [ ]:
CRs_trial1_control = cleaning.clean_dfs(
    dfs=CRs,
    this_var="zone-simple_cooling_requests",
    start_date=TRIAL_1_START,
    end_date=TRIAL_1_END,
    hourly_filter=control_filter,
    only_business_hours=True,
    no_weekends=True,
    remove_FCUs=True,
)

CRs_trial1_trial1 = cleaning.clean_dfs(
    dfs=CRs,
    this_var="zone-simple_cooling_requests",
    start_date=TRIAL_1_START,
    end_date=TRIAL_1_END,
    hourly_filter=trial1_filter,
    only_business_hours=True,
    no_weekends=True,
    remove_FCUs=True,
)

CRs_trial1_all = cleaning.clean_dfs(
    dfs=CRs,
    this_var="zone-simple_cooling_requests",
    start_date=TRIAL_1_START,
    end_date=TRIAL_1_END,
    hourly_filter=allzone_filter,
    only_business_hours=True,
    no_weekends=True,
    remove_FCUs=True,
)

CRs_trial1_control = base.run_passive_test_on_dfs(
    dfs=CRs_trial1_control, this_test="Mean", col_name="Control"
)
CRs_trial1_trial1 = base.run_passive_test_on_dfs(
    dfs=CRs_trial1_trial1, this_test="Mean", col_name="Trial 1"
)
CRs_trial1_all = base.run_passive_test_on_dfs(
    dfs=CRs_trial1_all, this_test="Mean", col_name="All-Zone"
)

CRs_combined_trial2 = base.combine_dicts(
    [CRs_trial1_control, CRs_trial1_trial1, CRs_trial1_all]
)

CRs_max_trial2 = {}
for project in PROJECTS:
    CRs_max_trial2[project] = CRs_combined_trial2[project].max(axis=1).to_frame()

In [ ]:
trial2_slices = {
    "OFF-2": [0.15],
    "OFF-7": [0.1],
    "OFF-4": [0.2],
    "OFF-5": [0.25],
    "OFF-3": [0.4],
    "OFF-6": [0.30],
}

In [ ]:
trial2_thresh_table = pd.DataFrame(
    index=PROJECTS,
    columns=[
        "Cooling Request Rate [Requests/Hour]",
    ],
)

for project in PROJECTS:
    trial2_thresh_table.loc[
        project, "Cooling Request Rate [Requests/Hour]"
    ] = trial2_slices[project][0]

In [ ]:
trial2 = clustering.run_1D_clustering_on_dict(
    CRs_max_trial2, trial2_slices, mapping={0: 1, 1: 0}
)

trial2["OFF-7"].loc["VAV5-05", 0] = 1  # manual adjustment

for project in PROJECTS:
    trial2[project].loc[EXCLUDED_ZONES_LISTS[project]["Any Trial"]] = 2

In [ ]:
# trial2_thresh_table

In [ ]:
fig = viz.make_dot_plot(
    y_data=CRs_max_trial2,
    color_data=trial2,
    color_legend={
        "name": {0: "Unadjusted in Trial 2", 1: "Adjusted in Trial 2", 2: "Excluded"},
        "color": {0: "Silver", 1: "DarkOrange", 2: "Dimgray"},
    },
    color_legend_order=[2, 0, 1],
    color_and_shape=True,
    shape_legend={
        "shape": {0: "circle", 1: "circle", 2: "x"},
    },
    y_axis_title="Cooling Request Rate<br>[Requests/Hr]",
    x_axis_title="Fraction of Zones [Unitless]",
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE,
    marker_size=MARKER_SIZE,
    horizontal_spacing=HORIZONTAL_SPACING,
    vertical_spacing=VERTICAL_SPACING,
    normalize_x=True,
)
fig = fig.update_layout(
    margin=dict(t=100),
    title=dict(
        text="Decisions for Trial 2",
        x=0.5,  # center horizontally
        xanchor="center",
        y=0.98,  # move up/down
        yanchor="top",
        font=dict(size=TITLE_SIZE),
    ),
    legend=dict(
        x=0.5,
        y=LEGEND_SPACING,
        xanchor="center",
        yanchor="top",
        orientation="h",
    ),
)

p = 0
r = 1
c = 1
for project in PROJECTS:
    if c == 3:
        r += 1
        c = 1
    fig.add_hline(
        y=trial2_slices[project][0],
        row=r,
        col=c,
        line_dash="dash",
        line_color="black",
        line_width=4,
    )

    c += 1
    p += 1

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/FigureB2.png")

## Trial 3

In [ ]:
CRs_trial2_control = cleaning.clean_dfs(
    dfs=CRs,
    this_var="zone-simple_cooling_requests",
    start_date=TRIAL_1_START,
    end_date=TRIAL_2_END,
    hourly_filter=control_filter,
    only_business_hours=True,
    no_weekends=True,
    remove_FCUs=True,
)

CRs_trial2_trial1 = cleaning.clean_dfs(
    dfs=CRs,
    this_var="zone-simple_cooling_requests",
    start_date=TRIAL_1_START,
    end_date=TRIAL_2_END,
    hourly_filter=trial1_filter,
    only_business_hours=True,
    no_weekends=True,
    remove_FCUs=True,
)

CRs_trial2_trial2 = cleaning.clean_dfs(
    dfs=CRs,
    this_var="zone-simple_cooling_requests",
    start_date=TRIAL_1_START,
    end_date=TRIAL_2_END,
    hourly_filter=trial2_filter,
    only_business_hours=True,
    no_weekends=True,
    remove_FCUs=True,
)

CRs_trial2_all = cleaning.clean_dfs(
    dfs=CRs,
    this_var="zone-simple_cooling_requests",
    start_date=TRIAL_1_START,
    end_date=TRIAL_2_END,
    hourly_filter=allzone_filter,
    only_business_hours=True,
    no_weekends=True,
    remove_FCUs=True,
)

CRs_trial2_control = base.run_passive_test_on_dfs(
    dfs=CRs_trial2_control, this_test="Mean", col_name="Control"
)
CRs_trial2_trial1 = base.run_passive_test_on_dfs(
    dfs=CRs_trial2_trial1, this_test="Mean", col_name="Trial 1"
)
CRs_trial2_trial2 = base.run_passive_test_on_dfs(
    dfs=CRs_trial2_trial2, this_test="Mean", col_name="Trial 1"
)
CRs_trial2_all = base.run_passive_test_on_dfs(
    dfs=CRs_trial2_all, this_test="Mean", col_name="All-Zone"
)

CRs_combined_trial3 = base.combine_dicts(
    [CRs_trial2_control, CRs_trial2_trial1, CRs_trial2_trial2, CRs_trial2_all]
)

CRs_max_trial3 = {}
for project in PROJECTS:
    CRs_max_trial3[project] = CRs_combined_trial3[project].max(axis=1).to_frame()

In [ ]:
trial3_slices = {
    "OFF-2": [0.6],
    "OFF-7": [0.43],
    "OFF-4": [0.60],
    "OFF-5": [0.68],
    "OFF-3": [0.8],
    "OFF-6": [0.80],
}

In [ ]:
these_projects = copy.deepcopy(PROJECTS)
these_projects.remove("OFF-7")

trial3_thresh_table = pd.DataFrame(
    index=these_projects,
    columns=[
        "Cooling Request Rate [Requests/Hour]",
    ],
)

for project in these_projects:
    trial3_thresh_table.loc[
        project, "Cooling Request Rate [Requests/Hour]"
    ] = trial3_slices[project][0]

In [ ]:
trial3 = clustering.run_1D_clustering_on_dict(
    CRs_max_trial3, trial3_slices, mapping={0: 1, 1: 0}
)

for project in PROJECTS:
    trial3[project].loc[EXCLUDED_ZONES_LISTS[project]["Any Trial"]] = 2

In [ ]:
# trial3_thresh_table

In [ ]:
fig = viz.make_dot_plot(
    y_data=CRs_max_trial3,
    color_data=trial3,
    color_legend={
        "name": {0: "Unadjusted in Trial 3", 1: "Adjusted in Trial 3", 2: "Excluded"},
        "color": {0: "Silver", 1: "FireBrick", 2: "Dimgray"},
    },
    color_legend_order=[2, 0, 1],
    color_and_shape=True,
    shape_legend={
        "shape": {0: "circle", 1: "circle", 2: "x"},
    },
    y_axis_title="Cooling Request Rate<br>[Requests/Hr]",
    x_axis_title="Fraction of Zones [Unitless]",
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE,
    marker_size=MARKER_SIZE,
    horizontal_spacing=HORIZONTAL_SPACING,
    vertical_spacing=VERTICAL_SPACING,
    normalize_x=True,
)
fig = fig.update_layout(
    margin=dict(t=100),
    title=dict(
        text="Decisions for Trial 3",
        x=0.5,  # center horizontally
        xanchor="center",
        y=0.98,  # move up/down
        yanchor="top",
        font=dict(size=TITLE_SIZE),
    ),
    legend=dict(
        x=0.5,
        y=LEGEND_SPACING,
        xanchor="center",
        yanchor="top",
        orientation="h",
    ),
)

p = 0
r = 1
c = 1
for project in PROJECTS:
    if c == 3:
        r += 1
        c = 1
    fig.add_hline(
        y=trial3_slices[project][0],
        row=r,
        col=c,
        line_dash="dash",
        line_color="black",
        line_width=4,
    )

    c += 1
    p += 1

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/FigureB3.png")

## CRs and deviation, control data

In [ ]:
(CRs_control, CRs_trial1, CRs_trial2, CRs_trial3, CRs_allzones) = get_trial_data(
    this_var="zone-simple_cooling_requests",
    this_var_clean="zone-dummy",
    start_date=pd.Timestamp("2024-05-01 00:00:00"),
    end_date=pd.Timestamp("2024-10-01 00:00:00"),
    only_business_hours=True,
    no_weekends=True,
    SI_units=True,
    this_test="Mean",
    clean_underlying=True,
)

In [ ]:
(
    deviation_control,
    deviation_trial1,
    deviation_trial2,
    deviation_trial3,
    deviation_allzones,
) = get_trial_data(
    this_var="zone-deviation_coolsp",
    this_var_clean="zone-deviation_coolsp",
    start_date=pd.Timestamp("2024-05-01 00:00:00"),
    end_date=pd.Timestamp("2024-10-01 00:00:00"),
    only_business_hours=True,
    no_weekends=True,
    SI_units=True,
    this_test="Mean",
    clean_underlying=True,
)

In [ ]:
CRs_control, deviation_control = base.make_common_index(
    [CRs_control, deviation_control]
)

In [ ]:
fig = viz.make_scatter_plot(
    y_data=CRs_control,
    x_data=deviation_control,
    color_data=ZONE_GROUPS,
    color_legend={
        "color": ZONE_GROUPS_NUM_TO_COLOR,
        "name": {
            0: "Increased in All-Zone",
            1: "Increased in Trial 1",
            2: "Increased in Trial 2",
            3: "Increased in Trial 3",
            4: "Excluded",
        },
    },
    color_and_shape=True,
    shape_legend={
        "shape": {
            0: "circle",
            1: "circle",
            2: "circle",
            3: "circle",
            4: "x",
        },
    },
    y_axis_title="Cooling Request Rate<br>[Requests/Hr]<br>Control Days",
    x_axis_title="T - CSP [°F] Control Days",
    # y_range=[0, 1.1],
    width=WIDTH,
    height=HEIGHT,
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE,
    marker_size=MARKER_SIZE + 1,
    horizontal_spacing=HORIZONTAL_SPACING,
    vertical_spacing=VERTICAL_SPACING,
    shape_legend_order=[],
)
fig = fig.update_layout(
    legend=dict(
        x=0.5,
        y=LEGEND_SPACING,
        xanchor="center",
        yanchor="top",
        orientation="h",
    ),
)

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/Figure2.png")

In [ ]:
CRs_control_tot = base.combine_to_total(CRs_control)
deviation_control_tot = base.combine_to_total(deviation_control)
zone_groups_tot = base.combine_to_total(ZONE_GROUPS)

In [ ]:
fig = viz.make_scatter_plot(
    y_data=CRs_control_tot,
    x_data=deviation_control_tot,
    color_data=zone_groups_tot,
    color_legend={
        "color": ZONE_GROUPS_NUM_TO_COLOR,
        "name": {
            0: "Increased in All-Zone",
            1: "Increased in Trial 1",
            2: "Increased in Trial 2",
            3: "Increased in Trial 3",
            4: "Excluded",
        },
    },
    color_and_shape=True,
    shape_legend={
        "shape": {
            0: "circle",
            1: "circle",
            2: "circle",
            3: "circle",
            4: "x",
        },
    },
    y_axis_title="Cooling Request Rate<br>[Requests/Hr]<br>Control Days",
    x_axis_title="T - CSP [°F] Control Days",
    # y_range=[0, 1.1],
    width=1200,
    height=550,
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE - 3,
    marker_size=MARKER_SIZE + 1,
    horizontal_spacing=HORIZONTAL_SPACING,
    vertical_spacing=VERTICAL_SPACING,
    shape_legend_order=[],
)
fig = fig.update_layout(
    legend=dict(
        x=1.05,
        y=-0.1,
        xanchor="left",
        yanchor="bottom",
        orientation="v",
    ),
)

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/GraphicalAbstract1.png")

## Lorenz curve

In [ ]:
(CRs_control, CRs_trial1, CRs_trial2, CRs_trial3, CRs_allzones) = get_trial_data(
    this_var="zone-simple_cooling_requests",
    this_var_clean="zone-dummy",
    start_date=pd.Timestamp("2024-05-01 00:00:00"),
    end_date=pd.Timestamp("2024-10-01 00:00:00"),
    only_business_hours=True,
    no_weekends=True,
    SI_units=True,
    this_test="Mean",
    clean_underlying=True,
)

In [ ]:
LORENZ = {}
for project in PROJECTS:
    LORENZ[project] = create_lorenz(CRs_control[project]).to_frame()

In [ ]:
fig = viz.make_dot_plot(
    y_data=LORENZ,
    color_data=ZONE_GROUPS,
    color_legend={
        "color": ZONE_GROUPS_NUM_TO_COLOR,
        "name": {
            0: "Increased In All-Zone",
            1: "Increased In Trial 1",
            2: "Increased In Trial 2",
            3: "Increased In Trial 3",
            4: "Excluded",
        },
    },
    color_and_shape=True,
    shape_legend={
        "shape": {
            0: "circle",
            1: "circle",
            2: "circle",
            3: "circle",
            4: "x",
        },
    },
    color_legend_order=[4, 0, 1, 2, 3],
    y_axis_title="Fraction of Cumulative<br>Cooling Requests<br>Control Days [Unitless]",
    x_axis_title="Fraction of Zones [Unitless]",
    y_range=[0, 1.1],
    normalize_x=True,
    width=WIDTH,
    height=HEIGHT,
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE,
    marker_size=MARKER_SIZE,
    horizontal_spacing=HORIZONTAL_SPACING,
    vertical_spacing=VERTICAL_SPACING,
)
fig = fig.update_layout(
    legend=dict(
        x=0.5,
        y=LEGEND_SPACING,
        xanchor="center",
        yanchor="top",
        orientation="h",
    ),
)

In [ ]:
p = 0
for i in range(1, 3):  # Columns
    for j in range(1, 4):  # Rows
        if p > 5:
            continue
        project = PROJECTS[p]
        fig.add_annotation(
            text=f"Trial 1: {int(round(len(ZONES_WRITTEN_TO_LISTS[project]['Trial 1']) / len(ALL_ZONES_LISTS[project]['VAVs']) * 100, 0))}%",
            xref=f"x{p+1}",
            yref=f"y{p+1}",
            x=0.6,
            y=0.6,
            xanchor="left",
            yanchor="top",
            showarrow=False,
            font=dict(size=ANNOTATION_SIZE, color="black"),
        )
        fig.add_annotation(
            text=f"Trial 2: {int(round(len(ZONES_WRITTEN_TO_LISTS[project]['Trial 2']) / len(ALL_ZONES_LISTS[project]['VAVs']) * 100, 0))}%",
            xref=f"x{p+1}",
            yref=f"y{p+1}",
            x=0.6,
            y=0.4,
            xanchor="left",
            yanchor="top",
            showarrow=False,
            font=dict(size=ANNOTATION_SIZE, color="black"),
        )
        if project not in ["OFF-7"]:
            fig.add_annotation(
                text=f"Trial 3: {int(round(len(ZONES_WRITTEN_TO_LISTS[project]['Trial 3']) / len(ALL_ZONES_LISTS[project]['VAVs']) * 100, 0))}%",
                xref=f"x{p+1}",
                yref=f"y{p+1}",
                x=0.6,
                y=0.2,
                xanchor="left",
                yanchor="top",
                showarrow=False,
                font=dict(size=ANNOTATION_SIZE, color="black"),
            )
        p += 1

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/Figure7.png")

# Cooling

In [ ]:
T = cleaning.clean_df(
    load_weather("2024"),
    "dummy",
    only_business_hours=ONLY_BUSINESS_HOURS,
    no_weekends=NO_WEEKENDS,
    start_date=SUMMER_START,
    end_date=SUMMER_END,
)

In [ ]:
cooling = load_utility_data(PROJECTS, (SUMMER_START, SUMMER_END), field="C")
cooling = cleaning.clean_by_column(
    df=cooling,
    only_business_hours=ONLY_BUSINESS_HOURS,
    no_weekends=NO_WEEKENDS,
    start_date=regression_functions.FORMAL_TRIALS_2024_START,
    end_date={
        key: value + pd.Timedelta(days=1)
        for key, value in regression_functions.FORMAL_TRIALS_2024_END.items()
    },
)
cooling = (cooling / constants.kW_to_ton) * 1000  # W
for project in PROJECTS:
    cooling[project] = cooling[project] / (SF[project] * base.M2_PER_SF)  # W/m2

T, cooling = base.trim_to_common_elements(
    [T, cooling], clean_cols=False, clean_idx=True
)

In [ ]:
abs_cooling_deltas_summary, abs_cooling_covs = get_building_wide_results(
    cooling, T, "Absolute Change", return_covs=True
)

(
    abs_cooling_deltas,
    abs_cooling_deltas_high,
    abs_cooling_deltas_low,
) = process_summary_df(abs_cooling_deltas_summary)

(
    abs_cooling_deltas_norm,
    abs_cooling_deltas_high_norm,
    abs_cooling_deltas_low_norm,
) = process_summary_df_norm(abs_cooling_deltas_summary, covs=abs_cooling_covs)

In [ ]:
# PORTION_ZONES

In [ ]:
# abs_cooling_deltas_norm

In [ ]:
# abs_cooling_deltas_summary[[
#    "Slope OAT", "Slope All-Zone", "Slope Trial 1", "Slope Trial 2", "Slope Trial 3", "Slope End Summer",
#    "Std Err OAT", "Std Err All-Zone", "Std Err Trial 1", "Std Err Trial 2", "Std Err Trial 3", "Std Err End Summer",
#    "P-Value OAT", "P-Value All-Zone", "P-Value Trial 1", "P-Value Trial 2", "P-Value Trial 3", "P-Value End Summer",
# ]].T

## More detail 

In [ ]:
fig = plot_2024_experiment_summary(
    deltas=-abs_cooling_deltas,
    deltas_high=-abs_cooling_deltas_high,
    deltas_low=-abs_cooling_deltas_low,
    portion_zones=PORTION_ZONES,
    y_axis_title="Absolute Cooling<br>Reduction [W/m<sup>2</sup>]",
    x_axis_title="Percent of Zones Adjusted [%]",
    y_range=[0, 120],
    x_range=[0, 100],
    mode="markers",
    error_thickness=2.5,
    whisker_len=10,
    append_zero=False,
    show_legend=False,
    width=WIDTH,
    height=HEIGHT,
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE,
    marker_size=MARKER_SIZE,
    horizontal_spacing=HORIZONTAL_SPACING,
    vertical_spacing=VERTICAL_SPACING,
)
fig = fig.update_layout(
    legend=dict(
        x=0.5,
        y=LEGEND_SPACING,
        xanchor="center",
        yanchor="top",
        orientation="h",
    ),
    # margin=dict(t=100)
)

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/Figure3B.png")

## normalized, power

In [ ]:
fig = viz.make_scatter_plot(
    y_data=abs_cooling_deltas_norm,
    # y_error_up_data=abs_cooling_deltas_high_norm,
    # y_error_down_data=abs_cooling_deltas_low_norm,
    x_data=PORTION_ZONES,
    y_axis_title="Relative Cooling Reduction<br>[% of All-Zones Trial]",
    x_axis_title="Percent of Zones Adjusted [%]",
    y_range=[0, 110],
    x_range=[0, 100],
    width=800,
    height=600,
    color_legend={
        "color": {
            "Trial 1": "Black",
            "Trial 2": "Black",
            "Trial 3": "Black",
            "All-Zone": "Black",
        },
    },
    dont_add_to_legend=["Trial 1", "Trial 2", "Trial 3", "All-Zone"],
    marker_size=12,
    text_size=28,
    legend_size=32,
)

In [ ]:
# extract values
cooling_y = abs_cooling_deltas_norm.values.flatten()  # dependent variable
portion_zones_x = PORTION_ZONES.values.flatten()  # independent variable

cooling_y = pd.to_numeric(pd.Series(cooling_y), errors="coerce").to_numpy()
portion_zones_x = pd.to_numeric(pd.Series(portion_zones_x), errors="coerce").to_numpy()

mask = (
    np.isfinite(portion_zones_x)
    & np.isfinite(cooling_y)
    & (portion_zones_x > 0)
    & (cooling_y > 0)
)
portion_zones_x = portion_zones_x[mask]
cooling_y = cooling_y[mask]

log_cooling_y = np.log(cooling_y)
log_x_intercept = sm.add_constant(np.log(portion_zones_x))

# run regression
cooling_pow_model = sm.OLS(log_cooling_y, log_x_intercept).fit()
cooling_intercept, cooling_p_pow = cooling_pow_model.params
cooling_c_pow = np.exp(cooling_intercept)
cooling_bias_correction = np.exp(0.5 * cooling_pow_model.mse_resid)

# predict
cooling_pred = cooling_pow_model.get_prediction(log_x_intercept).summary_frame(
    alpha=0.05
)
cooling_pred_mean = cooling_bias_correction * np.exp(cooling_pred["mean"])
cooling_pred_low = cooling_bias_correction * np.exp(cooling_pred["mean_ci_lower"])
cooling_pred_high = cooling_bias_correction * np.exp(cooling_pred["mean_ci_upper"])

log_x_intercept_for_plot = sm.add_constant(np.log(X_LINE_100))
cooling_for_plot = cooling_pow_model.get_prediction(
    log_x_intercept_for_plot
).summary_frame(alpha=0.05)
cooling_for_plot_mean = cooling_bias_correction * np.exp(cooling_for_plot["mean"])
cooling_for_plot_low = cooling_bias_correction * np.exp(
    cooling_for_plot["mean_ci_lower"]
)
cooling_for_plot_high = cooling_bias_correction * np.exp(
    cooling_for_plot["mean_ci_upper"]
)

# compute error
cooling_r2_logspace = cooling_pow_model.rsquared
ss_res = np.sum((cooling_y - cooling_pred_mean) ** 2)
ss_tot = np.sum((cooling_y - np.mean(cooling_y)) ** 2)
cooling_pow_r2 = 1 - ss_res / ss_tot

cooling_rmse_logspace = np.sqrt(cooling_pow_model.mse_resid)
cooling_pow_rmse = np.sqrt(np.mean((cooling_y - cooling_pred_mean) ** 2))

In [ ]:
# add to plot
fig = fig.add_trace(
    go.Scatter(
        x=list(X_LINE_100) + list(X_LINE_100)[::-1],
        y=list(cooling_for_plot_high) + list(cooling_for_plot_low)[::-1],
        fill="toself",
        fillcolor="rgba(0, 0, 0, 0.15)",
        line=dict(color="rgba(0,0,0,0)"),
        showlegend=False,
    ),
)

fig = fig.add_trace(
    go.Scatter(
        x=X_LINE_100,
        y=cooling_for_plot_mean,
        mode="lines",
        line=dict(color="black", dash="solid", width=5),
        showlegend=False,
    )
)

fig = fig.add_annotation(
    text=f"z = {cooling_c_pow:.2f}x<sup>{cooling_p_pow:.2f}</sup>",
    xref="x1",
    yref="y1",
    x=50,
    y=50,
    xanchor="left",
    yanchor="top",
    showarrow=False,
    font=dict(size=30, color="black"),
)

fig = fig.add_annotation(
    text="R<sup>2</sup>: {:.2f}".format(cooling_pow_r2),
    xref="x1",
    yref="y1",
    x=50,
    y=40,
    xanchor="left",
    yanchor="top",
    showarrow=False,
    font=dict(size=30, color="black"),
)

fig = fig.add_annotation(
    text=f"RMSE: {cooling_pow_rmse:.2f}",
    xref="x1",
    yref="y1",
    x=50,
    y=30,
    xanchor="left",
    yanchor="top",
    showarrow=False,
    font=dict(size=30, color="black"),
)

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/Figure3A.png")

# SAT

In [ ]:
T = cleaning.clean_df(
    load_weather("2024"),
    "dummy",
    only_business_hours=ONLY_BUSINESS_HOURS,
    no_weekends=NO_WEEKENDS,
    start_date=SUMMER_START,
    end_date=SUMMER_END,
)

In [ ]:
airflow = pull_from_dataset("2024", PROJECTS, "ahu-airflow")
airflow = cleaning.clean_dfs(
    dfs=airflow,
    this_var="ahu-airflow",
    start_date=SUMMER_START,
    end_date=SUMMER_END,
    only_business_hours=ONLY_BUSINESS_HOURS,
    no_weekends=NO_WEEKENDS,
    SI_units=True,
    resample_rule="1h",
)

dat = pull_from_dataset("2024", PROJECTS, "ahu-dat")
dat = cleaning.clean_dfs(
    dfs=dat,
    this_var="ahu-dat",
    start_date=SUMMER_START,
    end_date=SUMMER_END,
    only_business_hours=ONLY_BUSINESS_HOURS,
    no_weekends=NO_WEEKENDS,
    SI_units=True,
    resample_rule="1h",
)
dat = base.calculate_airflow_weighted_average(dat, airflow)

In [ ]:
abs_dat_deltas_summary, abs_dat_covs = get_building_wide_results(
    dat, T, "Absolute Change", return_covs=True
)
abs_dat_deltas, abs_dat_deltas_high, abs_dat_deltas_low = process_summary_df(
    abs_dat_deltas_summary
)
(
    abs_dat_deltas_norm,
    abs_dat_deltas_high_norm,
    abs_dat_deltas_low_norm,
) = process_summary_df_norm(abs_dat_deltas_summary, covs=abs_dat_covs)

In [ ]:
# abs_dat_deltas_summary[[
#    "Slope OAT", "Slope All-Zone", "Slope Trial 1", "Slope Trial 2", "Slope Trial 3", "Slope End Summer",
#    "Std Err OAT", "Std Err All-Zone", "Std Err Trial 1", "Std Err Trial 2", "Std Err Trial 3", "Std Err End Summer",
#    "P-Value OAT", "P-Value All-Zone", "P-Value Trial 1", "P-Value Trial 2", "P-Value Trial 3", "P-Value End Summer",
# ]].T

## More detail

In [ ]:
fig = plot_2024_experiment_summary(
    deltas=abs_dat_deltas,
    deltas_high=abs_dat_deltas_high,
    deltas_low=abs_dat_deltas_low,
    portion_zones=PORTION_ZONES,
    y_axis_title="Absolute SAT Increase [°C]",
    x_axis_title="Percent of Zones Adjusted [%]",
    # second_y_range=[0, 60],
    y_range=[-1, 7],
    x_range=[0, 100],
    mode="markers",
    error_thickness=2.5,
    whisker_len=10,
    append_zero=False,
    show_legend=False,
    # grid_color="white",
    # y_zerolinecolor="white",
    width=WIDTH,
    height=HEIGHT,
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE,
    marker_size=MARKER_SIZE,
    horizontal_spacing=HORIZONTAL_SPACING,
    vertical_spacing=VERTICAL_SPACING,
)
fig = fig.update_layout(
    legend=dict(
        x=0.5,
        y=LEGEND_SPACING,
        xanchor="center",
        yanchor="top",
        orientation="h",
    ),
    # margin=dict(t=100)
)

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/FigureA1.png")

## normalized, power

In [ ]:
fig = viz.make_scatter_plot(
    y_data=abs_dat_deltas_norm,
    # y_error_up_data=abs_dat_deltas_high_norm,
    # y_error_down_data=abs_dat_deltas_low_norm,
    x_data=PORTION_ZONES,
    y_axis_title="Relative SAT Increase<br>[% of All-Zones Trial]",
    x_axis_title="Percent of Zones Adjusted [%]",
    y_range=[-20, 120],
    x_range=[0, 100],
    width=800,
    height=600,
    color_legend={
        "color": {
            "Trial 1": "Black",
            "Trial 2": "Black",
            "Trial 3": "Black",
            "All-Zone": "Black",
        },
    },
    dont_add_to_legend=["Trial 1", "Trial 2", "Trial 3", "All-Zone"],
    marker_size=12,
    text_size=28,
    legend_size=32,
)

In [ ]:
# extract values
dat_y = abs_dat_deltas_norm.values.flatten()  # dependent variable
portion_zones_x = PORTION_ZONES.values.flatten()  # independent variable

dat_y = pd.to_numeric(pd.Series(dat_y), errors="coerce").to_numpy()
portion_zones_x = pd.to_numeric(pd.Series(portion_zones_x), errors="coerce").to_numpy()

mask = (
    np.isfinite(portion_zones_x)
    & np.isfinite(dat_y)
    & (portion_zones_x > 0)
    & (dat_y > 0)
)
portion_zones_x = portion_zones_x[mask]
dat_y = dat_y[mask]

log_dat_y = np.log(dat_y)
log_x_intercept = sm.add_constant(np.log(portion_zones_x))

# run regression
dat_pow_model = sm.OLS(log_dat_y, log_x_intercept).fit()
dat_intercept, dat_p_pow = dat_pow_model.params
dat_c_pow = np.exp(dat_intercept)
dat_bias_correction = np.exp(0.5 * dat_pow_model.mse_resid)

# predict
dat_pred = dat_pow_model.get_prediction(log_x_intercept).summary_frame(alpha=0.05)
dat_pred_mean = dat_bias_correction * np.exp(dat_pred["mean"])
dat_pred_low = dat_bias_correction * np.exp(dat_pred["mean_ci_lower"])
dat_pred_high = dat_bias_correction * np.exp(dat_pred["mean_ci_upper"])

log_x_intercept_for_plot = sm.add_constant(np.log(X_LINE_100))
dat_for_plot = dat_pow_model.get_prediction(log_x_intercept_for_plot).summary_frame(
    alpha=0.05
)
dat_for_plot_mean = dat_bias_correction * np.exp(dat_for_plot["mean"])
dat_for_plot_low = dat_bias_correction * np.exp(dat_for_plot["mean_ci_lower"])
dat_for_plot_high = dat_bias_correction * np.exp(dat_for_plot["mean_ci_upper"])

# compute error
dat_r2_logspace = dat_pow_model.rsquared
ss_res = np.sum((dat_y - dat_pred_mean) ** 2)
ss_tot = np.sum((dat_y - np.mean(dat_y)) ** 2)
dat_pow_r2 = 1 - ss_res / ss_tot

dat_rmse_logspace = np.sqrt(dat_pow_model.mse_resid)
dat_pow_rmse = np.sqrt(np.mean((dat_y - dat_pred_mean) ** 2))

In [ ]:
# add to plot
fig = fig.add_trace(
    go.Scatter(
        x=list(X_LINE_100) + list(X_LINE_100)[::-1],
        y=list(dat_for_plot_high) + list(dat_for_plot_low)[::-1],
        fill="toself",
        fillcolor="rgba(0, 0, 0, 0.15)",
        line=dict(color="rgba(0,0,0,0)"),
        showlegend=False,
    ),
)

fig = fig.add_trace(
    go.Scatter(
        x=X_LINE_100,
        y=dat_for_plot_mean,
        mode="lines",
        line=dict(color="black", dash="solid", width=5),
        showlegend=False,
    )
)

fig = fig.add_annotation(
    text=f"z = {dat_c_pow:.2f}x<sup>{dat_p_pow:.2f}</sup>",
    xref="x1",
    yref="y1",
    x=50,
    y=50,
    xanchor="left",
    yanchor="top",
    showarrow=False,
    font=dict(size=30, color="black"),
)

fig = fig.add_annotation(
    text="R<sup>2</sup>: {:.2f}".format(dat_pow_r2),
    xref="x1",
    yref="y1",
    x=50,
    y=40,
    xanchor="left",
    yanchor="top",
    showarrow=False,
    font=dict(size=30, color="black"),
)

fig = fig.add_annotation(
    text=f"RMSE: {dat_pow_rmse:.2f}",
    xref="x1",
    yref="y1",
    x=50,
    y=30,
    xanchor="left",
    yanchor="top",
    showarrow=False,
    font=dict(size=30, color="black"),
)

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/Figure4A.png")

# Airflow

In [ ]:
T = cleaning.clean_df(
    load_weather("2024"),
    "dummy",
    only_business_hours=ONLY_BUSINESS_HOURS,
    no_weekends=NO_WEEKENDS,
    start_date=SUMMER_START,
    end_date=SUMMER_END,
)

In [ ]:
airflow = pull_from_dataset("2024", PROJECTS, "zone-airflowsp")
airflow = cleaning.clean_dfs(
    dfs=airflow,
    this_var="zone-airflowsp",
    start_date=SUMMER_START,
    end_date=SUMMER_END,
    only_business_hours=ONLY_BUSINESS_HOURS,
    no_weekends=NO_WEEKENDS,
    SI_units=True,  # m3/hr
    resample_rule="1h",
)
for project in PROJECTS:
    airflow[project] = airflow[project].sum(axis=1).to_frame()
    airflow[project] = airflow[project] / (SF[project] * base.M2_PER_SF)  # m3/m2
    airflow[project].columns = [project]

In [ ]:
abs_airflow_deltas_summary, abs_airflow_covs = get_building_wide_results(
    airflow, T, "Absolute Change", return_covs=True
)
(
    abs_airflow_deltas,
    abs_airflow_deltas_high,
    abs_airflow_deltas_low,
) = process_summary_df(abs_airflow_deltas_summary)
(
    abs_airflow_deltas_norm,
    abs_airflow_deltas_high_norm,
    abs_airflow_deltas_low_norm,
) = process_summary_df_norm(abs_airflow_deltas_summary, covs=abs_airflow_covs)

In [ ]:
# abs_airflow_deltas_summary[[
#    "Slope OAT", "Slope All-Zone", "Slope Trial 1", "Slope Trial 2", "Slope Trial 3", "Slope End Summer",
#    "Std Err OAT", "Std Err All-Zone", "Std Err Trial 1", "Std Err Trial 2", "Std Err Trial 3", "Std Err End Summer",
#    "P-Value OAT", "P-Value All-Zone", "P-Value Trial 1", "P-Value Trial 2", "P-Value Trial 3", "P-Value End Summer",
# ]].T

## More detail

In [ ]:
fig = plot_2024_experiment_summary(
    deltas=-abs_airflow_deltas,
    deltas_high=-abs_airflow_deltas_high,
    deltas_low=-abs_airflow_deltas_low,
    portion_zones=PORTION_ZONES,
    y_axis_title="Absolute Airflow<br>Reduction [m<sup>3</sup>/Hr/m<sup>2</sup>]",
    x_axis_title="Percent of Zones Adjusted [%]",
    # second_y_range=[0, 60],
    y_range=[-0.5, 2.5],
    x_range=[0, 100],
    error_thickness=2.5,
    whisker_len=10,
    append_zero=False,
    show_legend=False,
    # grid_color="white",
    # y_zerolinecolor="white",
    mode="markers",
    width=WIDTH,
    height=HEIGHT,
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE,
    marker_size=MARKER_SIZE,
    horizontal_spacing=HORIZONTAL_SPACING,  # overrided
    vertical_spacing=VERTICAL_SPACING,
)
fig = fig.update_layout(
    legend=dict(
        x=0.5,
        y=-0.125,
        xanchor="center",
        yanchor="top",
        orientation="h",
    ),
    # margin=dict(t=100)
)

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/FigureA2.png")

## normalized, power

In [ ]:
fig = viz.make_scatter_plot(
    y_data=abs_airflow_deltas_norm,
    # y_error_up_data=abs_airflow_deltas_high_norm,
    # y_error_down_data=abs_airflow_deltas_low_norm,
    x_data=PORTION_ZONES,
    y_axis_title="Relative Airflow Reduction<br>[% of All-Zones Trial]",
    x_axis_title="Percent of Zones Adjusted [%]",
    y_range=[-20, 120],
    x_range=[0, 100],
    width=800,
    height=600,
    color_legend={
        "color": {
            "Trial 1": "Black",
            "Trial 2": "Black",
            "Trial 3": "Black",
            "All-Zone": "Black",
        },
    },
    dont_add_to_legend=["Trial 1", "Trial 2", "Trial 3", "All-Zone"],
    marker_size=12,
    text_size=28,
    legend_size=32,
)

In [ ]:
# extract values
airflow_y = abs_airflow_deltas_norm.values.flatten()  # dependent variable
portion_zones_x = PORTION_ZONES.values.flatten()  # independent variable

airflow_y = pd.to_numeric(pd.Series(airflow_y), errors="coerce").to_numpy()
portion_zones_x = pd.to_numeric(pd.Series(portion_zones_x), errors="coerce").to_numpy()

mask = (
    np.isfinite(portion_zones_x)
    & np.isfinite(airflow_y)
    & (portion_zones_x > 0)
    & (airflow_y > 0)
)
portion_zones_x = portion_zones_x[mask]
airflow_y = airflow_y[mask]

log_airflow_y = np.log(airflow_y)
log_x_intercept = sm.add_constant(np.log(portion_zones_x))

# run regression
airflow_pow_model = sm.OLS(log_airflow_y, log_x_intercept).fit()
airflow_intercept, airflow_p_pow = airflow_pow_model.params
airflow_c_pow = np.exp(airflow_intercept)
airflow_bias_correction = np.exp(0.5 * airflow_pow_model.mse_resid)

# predict
airflow_pred = airflow_pow_model.get_prediction(log_x_intercept).summary_frame(
    alpha=0.05
)
airflow_pred_mean = airflow_bias_correction * np.exp(airflow_pred["mean"])
airflow_pred_low = airflow_bias_correction * np.exp(airflow_pred["mean_ci_lower"])
airflow_pred_high = airflow_bias_correction * np.exp(airflow_pred["mean_ci_upper"])

log_x_intercept_for_plot = sm.add_constant(np.log(X_LINE_100))
airflow_for_plot = airflow_pow_model.get_prediction(
    log_x_intercept_for_plot
).summary_frame(alpha=0.05)
airflow_for_plot_mean = airflow_bias_correction * np.exp(airflow_for_plot["mean"])
airflow_for_plot_low = airflow_bias_correction * np.exp(
    airflow_for_plot["mean_ci_lower"]
)
airflow_for_plot_high = airflow_bias_correction * np.exp(
    airflow_for_plot["mean_ci_upper"]
)

# compute error
airflow_r2_logspace = airflow_pow_model.rsquared
ss_res = np.sum((airflow_y - airflow_pred_mean) ** 2)
ss_tot = np.sum((airflow_y - np.mean(airflow_y)) ** 2)
airflow_pow_r2 = 1 - ss_res / ss_tot

airflow_rmse_logspace = np.sqrt(airflow_pow_model.mse_resid)
airflow_pow_rmse = np.sqrt(np.mean((airflow_y - airflow_pred_mean) ** 2))

In [ ]:
# add to plot
fig = fig.add_trace(
    go.Scatter(
        x=list(X_LINE_100) + list(X_LINE_100)[::-1],
        y=list(airflow_for_plot_high) + list(airflow_for_plot_low)[::-1],
        fill="toself",
        fillcolor="rgba(0, 0, 0, 0.15)",
        line=dict(color="rgba(0,0,0,0)"),
        showlegend=False,
    ),
)

fig = fig.add_trace(
    go.Scatter(
        x=X_LINE_100,
        y=airflow_for_plot_mean,
        mode="lines",
        line=dict(color="black", dash="solid", width=5),
        showlegend=False,
    )
)

fig = fig.add_annotation(
    text=f"z = {airflow_c_pow:.2f}x<sup>{airflow_p_pow:.2f}</sup>",
    xref="x1",
    yref="y1",
    x=50,
    y=50,
    xanchor="left",
    yanchor="top",
    showarrow=False,
    font=dict(size=30, color="black"),
)

fig = fig.add_annotation(
    text="R<sup>2</sup>: {:.2f}".format(airflow_pow_r2),
    xref="x1",
    yref="y1",
    x=50,
    y=40,
    xanchor="left",
    yanchor="top",
    showarrow=False,
    font=dict(size=30, color="black"),
)

fig = fig.add_annotation(
    text=f"RMSE: {airflow_pow_rmse:.2f}",
    xref="x1",
    yref="y1",
    x=50,
    y=30,
    xanchor="left",
    yanchor="top",
    showarrow=False,
    font=dict(size=30, color="black"),
)

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/Figure4B.png")

# CRs

In [ ]:
(CRs_control, CRs_trial1, CRs_trial2, CRs_trial3, CRs_allzones) = get_trial_data(
    this_var="zone-simple_cooling_requests",
    this_var_clean="zone-dummy",
    start_date=pd.Timestamp("2024-05-01 00:00:00"),
    end_date=pd.Timestamp("2024-10-01 00:00:00"),
    only_business_hours=True,
    no_weekends=True,
    SI_units=True,
    this_test="Mean",
    clean_underlying=True,
)

## By group

In [ ]:
CRs_summary = {}
tot_zones = 0
frac_zones_tot = pd.Series(
    0,
    index=[
        "Excluded",
        "All-Zone",
        "Trial 1",
        "Trial 2",
        "Trial 3",
    ],
)

for project in PROJECTS:
    these_CR_results = pd.DataFrame(
        index=[
            "Frac of<br>Zones",
            "Control",
            "Trial 3",
            "Trial 2",
            "Trial 1",
            "All-Zone<br>Trial",
        ],
        columns=[
            "Excluded",
            "All-Zone",
            "Trial 1",
            "Trial 2",
            "Trial 3",
        ],
    )
    these_groups = ZONE_GROUPS[project]
    these_CRs = CRs_control[project]
    common = list(
        set(list(these_groups.index)).intersection(set(list(these_CRs.index)))
    )
    these_groups = these_groups.loc[common, :]

    tot_zones += len(common)

    for i in ZONE_GROUPS_NUM_TO_STR:
        these_zones = list(these_groups[these_groups == i].dropna().index)
        frac_zones_tot[ZONE_GROUPS_NUM_TO_STR[i]] += len(these_zones)

        # fraction of zones
        these_CR_results.loc["Frac of<br>Zones", ZONE_GROUPS_NUM_TO_STR[i]] = len(
            these_zones
        ) / len(common)
        # control
        these_CR_results.loc["Control", ZONE_GROUPS_NUM_TO_STR[i]] = (
            CRs_control[project].loc[these_zones, :].sum().iloc[0]
        )
        # trial 3
        these_CR_results.loc["Trial 3", ZONE_GROUPS_NUM_TO_STR[i]] = (
            CRs_trial3[project].loc[these_zones, :].sum().iloc[0]
        )
        # trial 2
        these_CR_results.loc["Trial 2", ZONE_GROUPS_NUM_TO_STR[i]] = (
            CRs_trial2[project].loc[these_zones, :].sum().iloc[0]
        )
        # trial 2
        these_CR_results.loc["Trial 1", ZONE_GROUPS_NUM_TO_STR[i]] = (
            CRs_trial1[project].loc[these_zones, :].sum().iloc[0]
        )
        # all zone
        these_CR_results.loc["All-Zone<br>Trial", ZONE_GROUPS_NUM_TO_STR[i]] = (
            CRs_allzones[project].loc[these_zones, :].sum().iloc[0]
        )

    CRs_summary[project] = these_CR_results

frac_zones_tot = frac_zones_tot / tot_zones

In [ ]:
fig = viz.make_bar_plot(
    y_data=CRs_summary,
    secondary_bars=["Control", "Trial 3", "Trial 2", "Trial 1", "All-Zone<br>Trial"],
    bar_mode="stack",
    bar_width=0.8,
    y_axis_title="Fraction of Zones<br>[Unitless]",
    secondary_y_axis_title="Cooling Request Rate<br>[Requests/Hr]",
    bar_legend={
        "color": {
            "Excluded": "Dimgray",
            "All-Zone": "Silver",
            "Trial 1": "ForestGreen",
            "Trial 2": "DarkOrange",
            "Trial 3": "Firebrick",
        },
        "name": {
            "Excluded": "Excluded",
            "All-Zone": "Increased In All-Zone",
            "Trial 1": "Increased In Trial 1",
            "Trial 2": "Increased In Trial 2",
            "Trial 3": "Increased In Trial 3",
        },
        "pattern": {
            "Excluded": "/",
            "All-Zone": "",
            "Trial 1": "",
            "Trial 2": "",
            "Trial 3": "",
        },
    },
    y_range=[0, 1],
    legend_order="forward",
    force_same_yaxes=False,
    legend_marker_size=20,
    width=WIDTH,
    height=HEIGHT,
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE,
    marker_size=MARKER_SIZE,
    horizontal_spacing=0.225,  # overrided
    vertical_spacing=VERTICAL_SPACING,
)
fig = fig.update_layout(
    legend=dict(
        x=0.5,
        y=LEGEND_SPACING,
        xanchor="center",
        yanchor="top",
        orientation="h",
    ),
    # margin=dict(t=100)
)
fig = fig.add_vline(x=0.5, line_width=4, line_color="black", line_dash="dash")

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/Figure5.png")

In [ ]:
CRs_summary_tot = pd.DataFrame(
    0,
    index=[
        "Frac of<br>Zones",
        "Control",
        "Trial 3",
        "Trial 2",
        "Trial 1",
        "All-Zone<br>Trial",
    ],
    columns=[
        "Excluded",
        "All-Zone",
        "Trial 1",
        "Trial 2",
        "Trial 3",
    ],
)
for project in PROJECTS:
    CRs_summary_tot += CRs_summary[project]
CRs_summary_tot.loc["Frac of<br>Zones", :] = frac_zones_tot

In [ ]:
fig = viz.make_bar_plot(
    y_data=CRs_summary_tot,
    secondary_bars=["Control", "Trial 3", "Trial 2", "Trial 1", "All-Zone<br>Trial"],
    bar_mode="stack",
    bar_width=0.8,
    y_axis_title="Fraction of Zones [Unitless]",
    secondary_y_axis_title="Cooling Request Rate<br>[Requests/Hr]",
    bar_legend={
        "color": {
            "Excluded": "Dimgray",
            "All-Zone": "Silver",
            "Trial 1": "ForestGreen",
            "Trial 2": "DarkOrange",
            "Trial 3": "Firebrick",
        },
        "name": {
            "Excluded": "Excluded",
            "All-Zone": "Increased In All-Zone",
            "Trial 1": "Increased In Trial 1",
            "Trial 2": "Increased In Trial 2",
            "Trial 3": "Increased In Trial 3",
        },
        "pattern": {
            "Excluded": "/",
            "All-Zone": "",
            "Trial 1": "",
            "Trial 2": "",
            "Trial 3": "",
        },
    },
    y_range=[0, 1],
    legend_order="forward",
    force_same_yaxes=False,
    legend_marker_size=20,
    width=WIDTH,
    height=700,
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE,
    marker_size=MARKER_SIZE,
    horizontal_spacing=0.225,  # overrided
    vertical_spacing=VERTICAL_SPACING,
)
fig = fig.update_layout(
    legend=dict(
        x=0.5,
        y=-0.2,
        xanchor="center",
        yanchor="top",
        orientation="h",
    ),
    # margin=dict(t=100)
)
fig = fig.add_vline(x=0.5, line_width=4, line_color="black", line_dash="dash")

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/GraphicalAbstract2.png")

# Temperature

In [ ]:
(temp_control, temp_trial1, temp_trial2, temp_trial3, temp_allzones) = get_trial_data(
    this_var="zone-temps",
    this_var_clean="zone-temps",
    start_date=pd.Timestamp("2024-05-01 00:00:00"),
    end_date=pd.Timestamp("2024-10-01 00:00:00"),
    only_business_hours=True,
    no_weekends=True,
    SI_units=True,
    this_test="Mean",
)

## Adjusted, Unadjusted

In [ ]:
temp_dicts = {
    "Control": temp_control,
    "Trial 1": temp_trial1,
    "Trial 2": temp_trial2,
    "Trial 3": temp_trial3,
    "All-Zone": temp_allzones,
}

In [ ]:
temp_tot = pd.DataFrame(
    0,
    index=[
        "Control",
        "Trial 3",
        "Trial 2",
        "Trial 1",
        "All-Zone",
    ],
    columns=[
        "CSP Unadjusted",
        "CSP Increased",
    ],
)

zones_tot = pd.DataFrame(
    0,
    index=[
        "Control",
        "Trial 3",
        "Trial 2",
        "Trial 1",
        "All-Zone",
    ],
    columns=[
        "CSP Unadjusted",
        "CSP Increased",
    ],
)

temp_summary2 = {}
for project in PROJECTS:
    these_temp_results = pd.DataFrame(
        index=[
            "Control",
            "Trial 3",
            "Trial 2",
            "Trial 1",
            "All-Zone",
        ],
        columns=[
            "CSP Unadjusted",
            "CSP Increased",
        ],
    )
    written_to_dict = copy.deepcopy(ZONES_WRITTEN_TO_LISTS[project])
    written_to_dict["Control"] = []

    for trial in these_temp_results.index:
        if project == "OFF-7" and trial == "Trial 3":
            continue
        all_zones = list(temp_dicts[trial][project].index)
        written_to = written_to_dict[trial]
        written_to = list(set(written_to).intersection(set(all_zones)))
        not_written_to = list(set(all_zones) - set(written_to))
        these_temp_results.loc[trial, "CSP Increased"] = (
            temp_dicts[trial][project].loc[written_to, :].mean().iloc[0]
        )
        these_temp_results.loc[trial, "CSP Unadjusted"] = (
            temp_dicts[trial][project].loc[not_written_to, :].mean().iloc[0]
        )

        # for total
        temp_tot.loc[trial, "CSP Increased"] += (
            temp_dicts[trial][project].loc[written_to, :].sum().iloc[0]
        )
        temp_tot.loc[trial, "CSP Unadjusted"] += (
            temp_dicts[trial][project].loc[not_written_to, :].sum().iloc[0]
        )
        zones_tot.loc[trial, "CSP Increased"] += len(written_to)
        zones_tot.loc[trial, "CSP Unadjusted"] += len(not_written_to)

    temp_summary2[project] = these_temp_results

In [ ]:
temp_tot = temp_tot.replace(0, np.nan)
zones_tot = zones_tot.replace(0, np.nan)
temp_tot = temp_tot / zones_tot

In [ ]:
fig = viz.make_bar_plot(
    y_data=temp_summary2,
    bar_mode="group",
    bar_width=0.8,
    y_axis_title="Temperature [°C]",
    bar_legend={
        "color": {
            "CSP Unadjusted": "SlateBlue",
            "CSP Increased": "Grey",
        },
    },
    y_range=[20, 25],
    legend_order="reversed",
    force_same_yaxes=False,
    legend_marker_size=22,
    width=WIDTH,
    height=HEIGHT,
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE,
    marker_size=MARKER_SIZE,
    horizontal_spacing=HORIZONTAL_SPACING,
    vertical_spacing=VERTICAL_SPACING,
)

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/Figure6.png")

In [ ]:
fig = viz.make_bar_plot(
    y_data=temp_tot,
    bar_mode="group",
    bar_width=0.8,
    y_axis_title="Temperature [°C]",
    bar_legend={
        "color": {
            "CSP Unadjusted": "SlateBlue",
            "CSP Increased": "Grey",
        },
    },
    y_range=[20, 25],
    legend_order="reversed",
    force_same_yaxes=False,
    legend_marker_size=22,
    width=850,
    height=550,
    num_cols=NUM_COLS,
    title_size=TITLE_SIZE,
    text_size=TEXT_SIZE,
    legend_size=LEGEND_SIZE,
    marker_size=MARKER_SIZE,
    horizontal_spacing=HORIZONTAL_SPACING,
    vertical_spacing=VERTICAL_SPACING,
)
fig = fig.update_layout(
    legend=dict(
        x=0.5,
        y=-0.15,
        xanchor="center",
        yanchor="top",
        orientation="h",
    ),
)

In [ ]:
# fig

In [ ]:
fig.write_image(f"{IMAGE_PATH}/GraphicalAbstract3.png")